# Combined Negotiation Analysis Notebook

## Part 1 — Model-Free Analysis Pipeline for Five Datasets  
Runs the existing model-free analysis workflow across:

1. Real human-to-human: `dealerships`  
2. Lab distributive human-to-human: `heddaya`  
3. Lab integrative human-to-human: `casino`  
4. Agent-to-agent distributive: `a2a`  
5. Agent-to-agent selling: `emaad`

## Part 2 — Dynamic Modeling Pipeline for Five Datasets  
Runs the existing Stage 1.5 dynamic modeling workflow on the same five datasets.

**Output layouts and plotting/reporting structure are preserved from the original notebooks.**  
Only the dataset configuration has been unified so both pipelines run in one file.


# PART 1 — Model-Free Analysis Pipeline

# Model-Free Negotiation Emotion Pipeline — Four Datasets
### Heddaya · Casino · Dealerships · Emaad

Runs the complete advisor-priority pipeline (uncertainty → model-free features → outcome validation)
across all four corpora simultaneously and produces:

- **Per-dataset outputs**: trajectory CI plots, feature CSVs, correlation tables, prediction models
- **Cross-dataset comparison tables** at every stage
- **Combined 27-panel buyer-vs-seller trajectory grids** matching the reference image format

Coverage: Modules 1–3 (uncertainty → descriptive → outcome validation).


In [23]:
!pip install -q numpy pandas scipy scikit-learn matplotlib seaborn

In [24]:
from __future__ import annotations

import os
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr, spearmanr, ttest_ind
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, r2_score, mean_absolute_error,
)
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
print("✓ Imports OK")


✓ Imports OK


## Configuration — edit paths here

In [25]:
BASE_DIR = Path("/content/optimal-nego")
OUTPUT_DIR = BASE_DIR / "outputs"

def resolve_input_path(*candidates):
    """Return first existing candidate; otherwise return the first candidate for transparent failure."""
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return str(p)
    return str(Path(candidates[0]))

DATASETS = {
    "dealerships": resolve_input_path(
        Path("/content/stage1_turns_dealership-nego.csv"),
        OUTPUT_DIR / "dealerships_nego" / "stage1_turns_embedded.csv",
        Path("/content/optimal-nego/outputs/dealerships/stage1_turns_embedded.csv"),
    ),
    "heddaya": resolve_input_path(
        Path("/content/stage1_turns_heddaya-nego.csv"),
        OUTPUT_DIR / "heddaya_nego" / "stage1_turns_embedded.csv",
        Path("/content/optimal-nego/outputs/heddaya/stage1_turns_embedded.csv"),
    ),
    "casino": resolve_input_path(
        Path("/content/stage1_turns_casino.csv"),
        OUTPUT_DIR / "casino_nego" / "stage1_turns_embedded.csv",
        Path("/content/optimal-nego/outputs/casino/stage1_turns_embedded.csv"),
    ),
    "a2a": resolve_input_path(
        Path("/content/stage1_turns_a2a-nego.csv"),
        OUTPUT_DIR / "a2a_nego" / "stage1_turns_embedded.csv",
        OUTPUT_DIR / "a2a_negotiations" / "stage1_turns_embedded.csv",
        Path("/content/stage1_turns_embedded.csv"),
        Path("/content/a2a_negotiations/stage1_turns_embedded.csv"),
        Path("/mnt/data/stage1_turns_embedded.csv"),
    ),
    "emaad": resolve_input_path(
        Path("/content/stage1_turns_emaad-sales.csv"),
        OUTPUT_DIR / "emaad_sales" / "stage1_turns_embedded.csv",
        Path("/content/optimal-nego/outputs/emaad/stage1_turns_embedded.csv"),
    ),
}

OUTPUT_ROOT = Path("/content/optimal-nego/outputs/model_free_multi_dataset")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Pipeline settings — reduce N_BOOT to 200 for fast iteration during development
N_BOOT   = 500
N_BINS   = 10
N_FOLDS  = 5
TOP_N    = 27   # show all 27 emotion panels in combined grid (matches reference image)

# Column constants
ID_COL   = "conversation_id"
TURN_COL = "turn_index"
ROLE_COL = "role"

# Dataset colours for cross-dataset plots
DS_COLORS = {
    "dealerships": "#E05A47",
    "heddaya":     "#1A8A7A",
    "casino":      "#7B68C8",
    "a2a":         "#3B82F6",
    "emaad":       "#E8A945",
}

# Role colours — blue buyer, orange seller (matches reference image)
ROLE_COLORS = {"buyer": "#4C9EEB", "seller": "#F0956A"}

GOEMOTIONS_27 = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval","disgust",
    "embarrassment","excitement","fear","gratitude","grief","joy","love",
    "nervousness","optimism","pride","realization","relief","remorse",
    "sadness","surprise",
]

BASE_NON_EMOTION_COLS = {
    "global_row_id","dataset_name","source_file","conversation_id","turn_index",
    "speaker_id","role","start_time","end_time","duration_min","text","outcome",
    "deal","is_deal","is_sale","sale","final_price","retail_price","wholesale_price",
    "budget_scenario","model_type","normalized_concession","concession_magnitude",
    "buyer_concession","seller_concession","satisfaction","trust",
    "willingness_to_return","outcome_binary","relative_time","dataset",
}

CANDIDATE_OUTCOMES = [
    "deal","is_deal","is_sale","sale","outcome_binary","final_price","normalized_concession",
    "concession_magnitude","buyer_concession","seller_concession",
    "satisfaction","trust","willingness_to_return",
]

print("✓ Configuration ready")
print(f"Output root: {OUTPUT_ROOT}")

✓ Configuration ready
Output root: /content/optimal-nego/outputs/model_free_multi_dataset


## Core functions — loading, bootstrapping, trajectories

In [26]:
# ─── LOADING ─────────────────────────────────────────────────────────────────

def load_stage1_data(input_path: str | Path) -> pd.DataFrame:
    """Load Stage 1 CSV and perform minimal validation."""
    df = pd.read_csv(input_path)
    required = [ID_COL, TURN_COL, ROLE_COL]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    df = df.sort_values([ID_COL, TURN_COL]).reset_index(drop=True)
    df[ROLE_COL] = df[ROLE_COL].astype(str).str.lower().str.strip()
    return df


def infer_deal_outcome(df: pd.DataFrame) -> pd.DataFrame:
    """Create binary outcome_binary from text outcome column if needed."""
    df = df.copy()
    if "outcome_binary" in df.columns:
        return df
    if "is_deal" in df.columns:
        df["outcome_binary"] = pd.to_numeric(df["is_deal"], errors="coerce")
        return df
    if "deal" in df.columns:
        df["outcome_binary"] = pd.to_numeric(df["deal"], errors="coerce")
        return df
    if "outcome" in df.columns:
        s = df["outcome"].astype(str).str.lower().str.strip()
        df["outcome_binary"] = np.nan
        df.loc[s.isin(["sale","deal","success","sold","yes","1"]), "outcome_binary"] = 1
        df.loc[s.isin(["no sale","no deal","failure","failed","no","0"]), "outcome_binary"] = 0
    return df


def detect_emotion_columns(df: pd.DataFrame) -> List[str]:
    """Detect GoEmotions/SST columns."""
    known = [c for c in GOEMOTIONS_27
             if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if known:
        return known
    return [c for c in df.columns
            if c not in BASE_NON_EMOTION_COLS
            and pd.api.types.is_numeric_dtype(df[c])]


def detect_outcome_columns(df: pd.DataFrame) -> List[str]:
    """Detect available outcome columns."""
    outcomes = []
    for c in CANDIDATE_OUTCOMES:
        if c in df.columns and df[c].notna().sum() > 0 and df[c].nunique(dropna=True) > 1:
            outcomes.append(c)
    if "outcome_binary" in outcomes and "deal" in outcomes:
        outcomes.remove("deal")
    if "outcome_binary" in outcomes and "is_deal" in outcomes:
        outcomes.remove("is_deal")
    return outcomes


def add_relative_time(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize each conversation to 0-1 time scale."""
    df = df.copy()
    def norm(x):
        d = max(1, x.max() - x.min())
        return (x - x.min()) / d
    df["relative_time"] = df.groupby(ID_COL)[TURN_COL].transform(norm)
    return df


# ─── BOOTSTRAP CI ────────────────────────────────────────────────────────────

def bootstrap_mean_ci(values: Iterable[float], n_boot: int = N_BOOT,
                      ci: float = 95, seed: int = 42) -> Tuple[float, float]:
    """Bootstrap confidence interval for a mean."""
    arr = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(arr) < 3:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot_means = np.array([rng.choice(arr, size=len(arr), replace=True).mean()
                           for _ in range(n_boot)])
    alpha = (100 - ci) / 2
    return float(np.percentile(boot_means, alpha)), float(np.percentile(boot_means, 100 - alpha))


# ─── TRAJECTORY FUNCTIONS ────────────────────────────────────────────────────

def binned_emotion_trajectory(df: pd.DataFrame, emotion: str,
                               bins: int = N_BINS, n_boot: int = N_BOOT) -> pd.DataFrame:
    """Binned mean trajectory with 95% bootstrap CI."""
    temp = df.copy()
    temp["time_bin"] = pd.cut(temp["relative_time"],
                               bins=np.linspace(0, 1, bins + 1),
                               labels=False, include_lowest=True)
    rows = []
    for b, g in temp.groupby("time_bin", dropna=True):
        lo, hi = bootstrap_mean_ci(g[emotion], n_boot=n_boot, seed=42 + int(b))
        rows.append({
            "emotion": emotion, "time_bin": int(b),
            "relative_time_mean": g["relative_time"].mean(),
            "mean": g[emotion].mean(), "ci_low": lo, "ci_high": hi,
            "n_turns": len(g), "n_conversations": g[ID_COL].nunique(),
        })
    return pd.DataFrame(rows)


def buyer_seller_gap_trajectory(df: pd.DataFrame, emotion: str,
                                 role_a: str = "buyer", role_b: str = "seller",
                                 bins: int = N_BINS, n_boot: int = N_BOOT) -> pd.DataFrame:
    """Buyer-seller emotion gap with bootstrapped CIs."""
    temp = df.copy()
    temp["time_bin"] = pd.cut(temp["relative_time"],
                               bins=np.linspace(0, 1, bins + 1),
                               labels=False, include_lowest=True)
    rng = np.random.default_rng(123)
    rows = []
    for b, g in temp.groupby("time_bin", dropna=True):
        a = g.loc[g[ROLE_COL] == role_a, emotion].dropna().to_numpy(dtype=float)
        s = g.loc[g[ROLE_COL] == role_b, emotion].dropna().to_numpy(dtype=float)
        if len(a) < 3 or len(s) < 3:
            gap, lo, hi = np.nan, np.nan, np.nan
        else:
            gap = float(a.mean() - s.mean())
            boot = [rng.choice(a, size=len(a), replace=True).mean()
                    - rng.choice(s, size=len(s), replace=True).mean()
                    for _ in range(n_boot)]
            lo, hi = float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))
        rows.append({
            "emotion": emotion, "time_bin": int(b),
            "relative_time_mean": g["relative_time"].mean(),
            "gap_role_a_minus_role_b": gap, "ci_low": lo, "ci_high": hi,
            "role_a": role_a, "role_b": role_b,
            "n_turns": len(g), "n_conversations": g[ID_COL].nunique(),
        })
    return pd.DataFrame(rows)


# ─── CONVERSATION-LEVEL FEATURES ─────────────────────────────────────────────

def conversation_level_features(df: pd.DataFrame, emotion_cols: List[str],
                                 outcomes: List[str]) -> pd.DataFrame:
    """Build conversation-level summaries for all emotions."""
    rows = []
    for cid, g in df.groupby(ID_COL, sort=False):
        g = g.sort_values(TURN_COL)
        row = {ID_COL: cid, "n_turns": len(g)}
        row["duration_observed"] = (
            g["end_time"].max() - g["start_time"].min()
            if {"start_time", "end_time"}.issubset(g.columns) else np.nan
        )
        for out in outcomes:
            if out in g.columns:
                row[out] = g[out].dropna().iloc[0] if g[out].notna().any() else np.nan
        for emo in emotion_cols:
            vals = g[emo].dropna().to_numpy(dtype=float)
            if len(vals) == 0:
                for sfx in ["mean","std","first","last","delta","min","max",
                             "early_mean","late_mean","late_minus_early",
                             "buyer_minus_seller_mean"]:
                    row[f"{emo}_{sfx}"] = np.nan
                continue
            row[f"{emo}_mean"]  = float(np.mean(vals))
            row[f"{emo}_std"]   = float(np.std(vals))
            row[f"{emo}_first"] = float(vals[0])
            row[f"{emo}_last"]  = float(vals[-1])
            row[f"{emo}_delta"] = float(vals[-1] - vals[0])
            row[f"{emo}_min"]   = float(np.min(vals))
            row[f"{emo}_max"]   = float(np.max(vals))
            early = g.loc[g["relative_time"] <= 0.33, emo].dropna()
            late  = g.loc[g["relative_time"] >= 0.67, emo].dropna()
            row[f"{emo}_early_mean"] = early.mean() if len(early) else np.nan
            row[f"{emo}_late_mean"]  = late.mean()  if len(late)  else np.nan
            row[f"{emo}_late_minus_early"] = (
                row[f"{emo}_late_mean"] - row[f"{emo}_early_mean"]
                if pd.notna(row[f"{emo}_late_mean"]) and pd.notna(row[f"{emo}_early_mean"])
                else np.nan
            )
            if {"buyer","seller"}.intersection(set(g[ROLE_COL].unique())):
                bv = g.loc[g[ROLE_COL]=="buyer", emo].dropna()
                sv = g.loc[g[ROLE_COL]=="seller", emo].dropna()
                row[f"{emo}_buyer_minus_seller_mean"] = (
                    bv.mean() - sv.mean() if len(bv) and len(sv) else np.nan
                )
        rows.append(row)
    return pd.DataFrame(rows)


def feature_columns_for_model(conv_df: pd.DataFrame, outcomes: List[str]) -> List[str]:
    exclude = {ID_COL, "n_turns", "duration_observed", "dataset"} | set(outcomes)
    return [c for c in conv_df.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(conv_df[c])]


print("✓ Core functions defined")


✓ Core functions defined


## Step 1 — Load all four datasets

In [27]:
all_data     = {}   # raw DataFrames
all_emotions = {}   # emotion column lists
all_outcomes = {}   # outcome column lists

for ds_name, path in DATASETS.items():
    if not Path(path).exists():
        print(f"  ✗ {ds_name}: not found at {path}")
        continue
    df = load_stage1_data(Path(path))
    df = infer_deal_outcome(df)
    df = add_relative_time(df)
    df["dataset"] = ds_name
    emo = detect_emotion_columns(df)
    out = detect_outcome_columns(df)
    all_data[ds_name]     = df
    all_emotions[ds_name] = emo
    all_outcomes[ds_name] = out
    print(f"  ✓ {ds_name:15s} | {df[ID_COL].nunique():4d} convs"
          f" | {len(df):6d} turns | {len(emo)} dims | outcomes={out}")

print(f"\n✓ Loaded {len(all_data)} datasets")


  ✓ dealerships     |   48 convs |   2480 turns | 27 dims | outcomes=['outcome_binary']
  ✓ heddaya         |  178 convs |   6384 turns | 27 dims | outcomes=[]
  ✓ casino          | 1030 convs |  14232 turns | 27 dims | outcomes=[]
  ✓ a2a             |  405 convs |   3332 turns | 27 dims | outcomes=['outcome_binary']
  ✓ emaad           |  200 convs |   3414 turns | 27 dims | outcomes=['outcome_binary']

✓ Loaded 5 datasets


## Step 2 — Cross-dataset summary table

In [28]:
rows = []
for ds, df in all_data.items():
    rows.append({
        "Dataset":          ds,
        "Type":             {"dealerships":"Distributive/Real","heddaya":"Distributive/Real",
                             "casino":"Integrative/Lab","emaad":"Scripted/Sales"}.get(ds,"—"),
        "Conversations":    df[ID_COL].nunique(),
        "Total turns":      len(df),
        "Avg turns/conv":   round(len(df)/df[ID_COL].nunique(), 1),
        "Emotion dims":     len(all_emotions[ds]),
        "Buyer turns":      (df[ROLE_COL]=="buyer").sum(),
        "Seller turns":     (df[ROLE_COL]=="seller").sum(),
        "Outcomes":         ", ".join(all_outcomes.get(ds,[])) or "none",
    })

summary_table = pd.DataFrame(rows).set_index("Dataset")
summary_table.to_csv(OUTPUT_ROOT / "00_dataset_summary.csv")
print(summary_table.to_string())


                          Type  Conversations  Total turns  Avg turns/conv  Emotion dims  Buyer turns  Seller turns        Outcomes
Dataset                                                                                                                            
dealerships  Distributive/Real             48         2480            51.7            27         1233          1247  outcome_binary
heddaya      Distributive/Real            178         6384            35.9            27         3243          3141            none
casino         Integrative/Lab           1030        14232            13.8            27         7107          7125            none
a2a                          —            405         3332             8.2            27         1857          1475  outcome_binary
emaad           Scripted/Sales            200         3414            17.1            27         1857          1557  outcome_binary


## Step 3 — Emotion activation: cross-dataset table and heatmap

In [29]:
# ─── Activation table ───
act_rows = {}
for ds, df in all_data.items():
    emo = all_emotions[ds]
    act_rows[ds] = df[emo].mean()

activation_table = pd.DataFrame(act_rows)
activation_table.index.name = "emotion"
# Sort by mean across datasets
activation_table["__mean__"] = activation_table.mean(axis=1)
activation_table = activation_table.sort_values("__mean__", ascending=False).drop(columns="__mean__")
activation_table.to_csv(OUTPUT_ROOT / "01_activation_table.csv")

print("Emotion activation by dataset (mean per turn):")
print(activation_table.round(4).to_string())

# ─── Heatmap ───
fig, ax = plt.subplots(figsize=(max(6, len(all_data)*2.5+2), 11))
sns.heatmap(activation_table, cmap="YlOrRd", annot=True, fmt=".4f",
            ax=ax, linewidths=0.4, linecolor="white",
            cbar_kws={"label": "Mean activation per turn"})
ax.set_title("Emotion Activation Across Datasets\n(all 27 dims, mean per turn)", fontsize=13)
ax.set_xlabel("Dataset"); ax.set_ylabel("Emotion")
plt.tight_layout()
fig.savefig(OUTPUT_ROOT / "01_activation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✓ Saved: 01_activation_heatmap.png")


Emotion activation by dataset (mean per turn):
                dealerships  heddaya  casino     a2a   emaad
emotion                                                     
curiosity            0.1658   0.0488  0.1414  0.2880  0.2012
approval             0.1762   0.1444  0.1421  0.1454  0.2024
admiration           0.0784   0.0663  0.0751  0.3594  0.1243
gratitude            0.0415   0.0140  0.0356  0.3031  0.1803
confusion            0.0579   0.0385  0.0347  0.0311  0.0584
optimism             0.0399   0.0259  0.0492  0.0759  0.0235
disapproval          0.0490   0.0371  0.0233  0.0198  0.0213
desire               0.0227   0.0192  0.0478  0.0353  0.0093
excitement           0.0184   0.0126  0.0279  0.0440  0.0187
joy                  0.0112   0.0109  0.0299  0.0337  0.0234
caring               0.0174   0.0076  0.0243  0.0295  0.0266
love                 0.0129   0.0178  0.0186  0.0168  0.0075
realization          0.0163   0.0200  0.0126  0.0085  0.0148
annoyance            0.0205   0.0099  

In [30]:
# ─── Emotion rank comparison table ───
rank_table = activation_table.rank(ascending=False).astype(int)
rank_table.to_csv(OUTPUT_ROOT / "01_activation_rank_table.csv")

print("\nActivation RANK by dataset (1 = most activated):")
print(rank_table.to_string())

print("\nAnger rank:")
for ds in all_data:
    if ds in rank_table.columns and "anger" in rank_table.index:
        print(f"  {ds:15s}: rank {rank_table.loc['anger', ds]} / {len(rank_table)}")



Activation RANK by dataset (1 = most activated):
                dealerships  heddaya  casino  a2a  emaad
emotion                                                 
curiosity                 2        3       2    3      2
approval                  1        1       1    4      1
admiration                3        2       3    1      4
gratitude                 6       11       6    2      3
confusion                 4        4       7    9      5
optimism                  7        6       4    5      7
disapproval               5        5      11   11      9
desire                    8        8       5    7     12
excitement               10       12       9    6     10
joy                      16       14       8    8      8
caring                   11       17      10   10      6
love                     15        9      12   12     13
realization              12        7      15   16     11
annoyance                 9       15      17   18     14
disappointment           14       13  

## Step 4 — 27-panel buyer vs seller trajectory grids
One grid per dataset (matches reference image format).  
Blue = buyer, orange = seller, shading = 95% bootstrap CI.

In [31]:
def plot_27panel_grid(df, emotion_cols, dataset_name, out_path, n_boot=N_BOOT, bins=N_BINS):
    """
    Plot all emotion dimensions as a combined grid.
    Each panel: buyer (blue) and seller (orange) mean trajectories with 95% CI.
    Matches the format of the uploaded reference image.
    """
    n_emo  = len(emotion_cols)
    n_cols = 6
    n_rows = (n_emo + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols*3.8, n_rows*3.0),
                              constrained_layout=True)
    axes = np.array(axes).flatten()

    for ax_idx, emo in enumerate(emotion_cols):
        ax = axes[ax_idx]
        has_data = False

        for role in ["buyer", "seller"]:
            sub = df[df[ROLE_COL] == role]
            if len(sub) < 5:
                continue
            traj = binned_emotion_trajectory(sub, emo, bins=bins, n_boot=n_boot)
            if traj.empty:
                continue
            color = ROLE_COLORS[role]
            ax.plot(traj["relative_time_mean"], traj["mean"],
                    marker="o", markersize=3, linewidth=1.6,
                    color=color, label=role.capitalize())
            ax.fill_between(traj["relative_time_mean"],
                            traj["ci_low"], traj["ci_high"],
                            alpha=0.2, color=color,
                            label=f"{role.capitalize()} CI")
            has_data = True

        if not has_data:
            ax.set_visible(False)
            continue

        ax.set_title(f"Buyer vs. Seller Emotion Trajectory with 95% CI: {emo}",
                     fontsize=7.5, pad=3)
        ax.set_xlabel("Normalized conversation time", fontsize=6.5)
        ax.set_ylabel("Mean activation", fontsize=6.5)
        ax.tick_params(labelsize=6)
        ax.grid(True, alpha=0.3, linewidth=0.4)
        ax.legend(fontsize=5.5, loc="upper right", framealpha=0.6,
                  handlelength=1.0, borderpad=0.4)

    for ax in axes[n_emo:]:
        ax.set_visible(False)

    fig.suptitle(
        f"Buyer vs. Seller Emotion Trajectory with 95% CI\n{dataset_name}",
        fontsize=13, fontweight="bold"
    )
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {out_path.name}")


print("Generating 27-panel grids for each dataset...")
for ds, df in all_data.items():
    ds_dir = OUTPUT_ROOT / ds
    ds_dir.mkdir(exist_ok=True)
    plot_27panel_grid(
        df, all_emotions[ds], ds,
        ds_dir / f"{ds}_role_based_emo_plots.png"
    )
print("\n✓ All 27-panel grids complete.")


Generating 27-panel grids for each dataset...
  ✓ dealerships_role_based_emo_plots.png
  ✓ heddaya_role_based_emo_plots.png
  ✓ casino_role_based_emo_plots.png
  ✓ a2a_role_based_emo_plots.png
  ✓ emaad_role_based_emo_plots.png

✓ All 27-panel grids complete.


## Step 5 — Cross-dataset trajectory comparison (top emotions)

In [32]:
# Determine globally top 8 emotions by mean activation across all datasets
global_means = pd.concat(
    [all_data[ds][all_emotions[ds]].mean().rename(ds) for ds in all_data],
    axis=1
).mean(axis=1).sort_values(ascending=False)
top8 = global_means.head(8).index.tolist()
print(f"Top 8 emotions globally: {top8}")

ds_list = list(all_data.keys())
n_ds = len(ds_list)

fig, axes = plt.subplots(len(top8), n_ds,
                          figsize=(n_ds*4.2, len(top8)*3.0),
                          constrained_layout=True)
if len(top8) == 1: axes = axes.reshape(1, -1)
if n_ds == 1: axes = axes.reshape(-1, 1)

for row_i, emo in enumerate(top8):
    for col_j, ds in enumerate(ds_list):
        ax = axes[row_i, col_j]
        df = all_data[ds]
        if emo not in all_emotions[ds]:
            ax.set_visible(False)
            continue

        for role in ["buyer", "seller"]:
            sub = df[df[ROLE_COL] == role]
            if len(sub) < 5:
                continue
            traj = binned_emotion_trajectory(sub, emo, bins=N_BINS, n_boot=N_BOOT)
            color = ROLE_COLORS[role]
            ax.plot(traj["relative_time_mean"], traj["mean"],
                    marker="o", markersize=3, linewidth=1.5,
                    color=color, label=role.capitalize())
            ax.fill_between(traj["relative_time_mean"],
                            traj["ci_low"], traj["ci_high"],
                            alpha=0.18, color=color)

        ax.set_title(f"{ds.upper()}\n{emo}", fontsize=8, fontweight="bold")
        ax.set_xlabel("Time (norm.)", fontsize=6.5)
        ax.set_ylabel("Activation", fontsize=6.5)
        ax.tick_params(labelsize=6)
        ax.grid(True, alpha=0.3, linewidth=0.4)
        if row_i == 0:
            ax.legend(fontsize=6.5, framealpha=0.6)

fig.suptitle("Top 8 Emotions: Buyer vs Seller Trajectories Across Datasets\n"
             "(Blue = Buyer, Orange = Seller, Shading = 95% CI)",
             fontsize=11, fontweight="bold")
fig.savefig(OUTPUT_ROOT / "02_cross_dataset_top8_trajectories.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("✓ Saved: 02_cross_dataset_top8_trajectories.png")


Top 8 emotions globally: ['curiosity', 'approval', 'admiration', 'gratitude', 'confusion', 'optimism', 'disapproval', 'desire']
✓ Saved: 02_cross_dataset_top8_trajectories.png


## Step 6 — Buyer-seller asymmetry table (cross-dataset)

In [33]:
from scipy.stats import ttest_ind

asym_rows = []
for ds, df in all_data.items():
    buyer  = df[df[ROLE_COL]=="buyer"]
    seller = df[df[ROLE_COL]=="seller"]
    for emo in all_emotions[ds]:
        bm = buyer[emo].mean()
        sm = seller[emo].mean()
        try:
            _, p = ttest_ind(buyer[emo].dropna(), seller[emo].dropna(), equal_var=False)
        except Exception:
            p = np.nan
        asym_rows.append({
            "dataset": ds, "emotion": emo,
            "buyer_mean": round(bm, 5), "seller_mean": round(sm, 5),
            "delta_buyer_minus_seller": round(bm - sm, 5),
            "abs_delta": round(abs(bm - sm), 5),
            "p_value": round(p, 4) if pd.notna(p) else np.nan,
            "significant_p05": "yes" if pd.notna(p) and p < 0.05 else "no",
        })

asym_df = pd.DataFrame(asym_rows)
asym_df.to_csv(OUTPUT_ROOT / "03_buyer_seller_asymmetry.csv", index=False)

# Print top 5 per dataset
for ds in all_data:
    sub = asym_df[asym_df["dataset"]==ds].nlargest(5, "abs_delta")
    print(f"\n{ds} — top 5 buyer-seller gaps:")
    print(sub[["emotion","buyer_mean","seller_mean","delta_buyer_minus_seller","p_value"]].to_string(index=False))



dealerships — top 5 buyer-seller gaps:
   emotion  buyer_mean  seller_mean  delta_buyer_minus_seller  p_value
admiration     0.09169      0.06521                   0.02648   0.0012
 gratitude     0.03003      0.05275                  -0.02272   0.0014
    desire     0.02738      0.01799                   0.00939   0.0025
    caring     0.01285      0.02185                  -0.00900   0.0004
   remorse     0.01207      0.02001                  -0.00795   0.0447

heddaya — top 5 buyer-seller gaps:
    emotion  buyer_mean  seller_mean  delta_buyer_minus_seller  p_value
   approval     0.13461      0.15449                  -0.01988   0.0000
 admiration     0.05779      0.07508                  -0.01729   0.0003
  curiosity     0.05278      0.04478                   0.00800   0.0188
disapproval     0.03408      0.04026                  -0.00618   0.0417
       love     0.02077      0.01468                   0.00609   0.0105

casino — top 5 buyer-seller gaps:
    emotion  buyer_mean  seller

In [34]:
# Max asymmetry per dataset — visual summary
max_asym = asym_df.groupby("dataset")["abs_delta"].max().reindex(list(all_data.keys()))

fig, ax = plt.subplots(figsize=(8, 5))
colors_list = [DS_COLORS.get(ds, "#888") for ds in max_asym.index]
bars = ax.bar(max_asym.index, max_asym.values, color=colors_list, alpha=0.85)
ax.set_ylabel("Max |buyer - seller| delta", fontsize=10)
ax.set_title("Maximum Buyer-Seller Emotional Asymmetry by Dataset\n"
             "(lower = more symmetric roles)", fontsize=11)
for bar, val in zip(bars, max_asym.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001,
            f"{val:.4f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
fig.savefig(OUTPUT_ROOT / "03_buyer_seller_asymmetry_summary.png", dpi=150)
plt.close()
print("✓ Saved: 03_buyer_seller_asymmetry_summary.png")


✓ Saved: 03_buyer_seller_asymmetry_summary.png


## Step 7 — Conversation-level features (model-free evidence)

In [35]:
all_conv = {}

for ds, df in all_data.items():
    ds_dir = OUTPUT_ROOT / ds
    ds_dir.mkdir(exist_ok=True)
    outs = all_outcomes[ds]
    emo  = all_emotions[ds]
    conv = conversation_level_features(df, emo, outs)
    conv["dataset"] = ds
    conv.to_csv(ds_dir / "conversation_features.csv", index=False)
    all_conv[ds] = conv
    print(f"  ✓ {ds}: {len(conv)} conversations, {len(conv.columns)} features")

print("\n✓ Conversation features built for all datasets.")


  ✓ dealerships: 48 conversations, 302 features
  ✓ heddaya: 178 conversations, 301 features
  ✓ casino: 1030 conversations, 301 features
  ✓ a2a: 405 conversations, 302 features
  ✓ emaad: 200 conversations, 302 features

✓ Conversation features built for all datasets.


In [36]:
all_conv = {}

for ds, df in all_data.items():
    ds_dir = OUTPUT_ROOT / ds
    ds_dir.mkdir(exist_ok=True)
    outs = all_outcomes[ds]
    emo  = all_emotions[ds]
    conv = conversation_level_features(df, emo, outs)
    conv["dataset"] = ds
    conv.to_csv(ds_dir / "conversation_features.csv", index=False)
    all_conv[ds] = conv
    print(f"  ✓ {ds}: {len(conv)} conversations, {len(conv.columns)} features")

print("\n✓ Conversation features built for all datasets.")


  ✓ dealerships: 48 conversations, 302 features
  ✓ heddaya: 178 conversations, 301 features
  ✓ casino: 1030 conversations, 301 features
  ✓ a2a: 405 conversations, 302 features
  ✓ emaad: 200 conversations, 302 features

✓ Conversation features built for all datasets.


In [37]:
# Cross-dataset descriptive: mean activation ranked per dataset
desc_rows = []
for ds, df in all_data.items():
    emo   = all_emotions[ds]
    means = df[emo].mean()
    ranks = means.rank(ascending=False)
    for e in emo:
        desc_rows.append({
            "dataset": ds, "emotion": e,
            "mean_activation":  round(float(means[e]), 5),
            "activation_rank":  int(ranks[e]),
            "std_activation":   round(float(df[e].std()), 5),
            "pct_nonzero":      round(float((df[e] > 0).mean() * 100), 2),
        })

desc_df = pd.DataFrame(desc_rows)
desc_df.to_csv(OUTPUT_ROOT / "04_descriptive_stats.csv", index=False)

# Pivot for readable cross-dataset comparison
pivot_act  = desc_df.pivot(index="emotion", columns="dataset", values="mean_activation")
pivot_rank = desc_df.pivot(index="emotion", columns="dataset", values="activation_rank")
pivot_act["cross_mean"] = pivot_act.mean(axis=1)
pivot_act  = pivot_act.sort_values("cross_mean", ascending=False).drop(columns="cross_mean")
pivot_rank = pivot_rank.reindex(pivot_act.index)

print("\nMean activation pivot:")
print(pivot_act.round(4).to_string())
print("\nActivation rank pivot:")
print(pivot_rank.to_string())



Mean activation pivot:
dataset            a2a  casino  dealerships   emaad  heddaya
emotion                                                     
curiosity       0.2880  0.1414       0.1658  0.2012   0.0488
approval        0.1454  0.1421       0.1762  0.2024   0.1444
admiration      0.3594  0.0751       0.0784  0.1243   0.0663
gratitude       0.3031  0.0356       0.0415  0.1803   0.0140
confusion       0.0311  0.0347       0.0579  0.0584   0.0384
optimism        0.0759  0.0492       0.0399  0.0235   0.0259
disapproval     0.0198  0.0233       0.0490  0.0213   0.0371
desire          0.0353  0.0478       0.0227  0.0093   0.0192
excitement      0.0440  0.0279       0.0184  0.0187   0.0126
joy             0.0337  0.0299       0.0112  0.0234   0.0109
caring          0.0295  0.0243       0.0174  0.0266   0.0076
love            0.0168  0.0186       0.0129  0.0074   0.0178
realization     0.0085  0.0126       0.0163  0.0148   0.0200
annoyance       0.0060  0.0112       0.0205  0.0072   0.0099


## Step 8 — Outcome-based validation: correlations

In [38]:
def run_correlations(conv_df, outcomes):
    """Pearson + Spearman correlations of emotion features with outcomes."""
    exclude = {ID_COL, "n_turns", "duration_observed", "dataset"} | set(outcomes)
    feat_cols = [c for c in conv_df.columns
                 if c not in exclude and pd.api.types.is_numeric_dtype(conv_df[c])]
    rows = []
    for out in outcomes:
        if out not in conv_df.columns:
            continue
        y = pd.to_numeric(conv_df[out], errors="coerce")
        for f in feat_cols:
            x = pd.to_numeric(conv_df[f], errors="coerce")
            sub = pd.DataFrame({"x": x, "y": y}).dropna()
            if len(sub) < 8 or sub["x"].nunique() < 2 or sub["y"].nunique() < 2:
                continue
            try:
                pr, pp = pearsonr(sub["x"], sub["y"])
            except Exception:
                pr, pp = np.nan, np.nan
            try:
                sr, sp = spearmanr(sub["x"], sub["y"])
            except Exception:
                sr, sp = np.nan, np.nan
            rows.append({
                "outcome": out, "feature": f, "n": len(sub),
                "pearson_r": round(pr,4) if pd.notna(pr) else np.nan,
                "pearson_p": round(pp,4) if pd.notna(pp) else np.nan,
                "spearman_r": round(sr,4) if pd.notna(sr) else np.nan,
                "spearman_p": round(sp,4) if pd.notna(sp) else np.nan,
                "abs_pearson_r": round(abs(pr),4) if pd.notna(pr) else np.nan,
            })
    result = pd.DataFrame(rows)
    if not result.empty:
        result = result.sort_values(["outcome","abs_pearson_r"], ascending=[True,False])
    return result


# Force 'outcome_binary' for 'a2a' if it's missing in all_outcomes and the column exists and has variance
if 'a2a' in all_outcomes and not all_outcomes['a2a']:
    a2a_df = all_data.get('a2a')
    if a2a_df is not None and 'outcome_binary' in a2a_df.columns:
        if a2a_df['outcome_binary'].notna().sum() > 0 and a2a_df['outcome_binary'].nunique(dropna=True) > 1:
            all_outcomes['a2a'] = ['outcome_binary']
            print(f"  Note: Forcing 'outcome_binary' as outcome for 'a2a' dataset.")
        else:
            print(f"  Note: 'outcome_binary' in 'a2a' has no variance or all NaNs, cannot use for correlation.")

all_corr = {}
for ds, conv in all_conv.items():
    outs = all_outcomes[ds]
    if not outs:
        print(f"  {ds}: no outcomes — skipping")
        continue
    corr = run_correlations(conv, outs)
    corr["dataset"] = ds
    corr.to_csv(OUTPUT_ROOT / ds / "correlations.csv", index=False)
    all_corr[ds] = corr
    n_sig = (corr["pearson_p"] < 0.05).sum() if not corr.empty else 0
    print(f"  ✓ {ds}: {len(corr)} pairs, {n_sig} significant (p<0.05)")

if all_corr:
    pd.concat(all_corr.values(), ignore_index=True).to_csv(
        OUTPUT_ROOT / "05_all_correlations.csv", index=False)
    print("\n✓ Combined correlation table saved.")

  ✓ dealerships: 297 pairs, 37 significant (p<0.05)
  heddaya: no outcomes — skipping
  casino: no outcomes — skipping
  ✓ a2a: 297 pairs, 206 significant (p<0.05)
  ✓ emaad: 297 pairs, 182 significant (p<0.05)

✓ Combined correlation table saved.


In [39]:
if all_corr:
    n_cols = len(all_corr)
    fig, axes = plt.subplots(1, n_cols, figsize=(6.5*n_cols, 8), constrained_layout=True)
    if n_cols == 1:
        axes = [axes]

    for ax, (ds, corr) in zip(axes, all_corr.items()):
        first_out = corr["outcome"].iloc[0]
        sub = (corr[corr["outcome"] == first_out]
               .dropna(subset=["pearson_r"])
               .head(20))
        colors_bar = [DS_COLORS.get(ds,"#888") if r >= 0 else "#E05A47"
                      for r in sub["pearson_r"]]
        ax.barh(sub["feature"], sub["pearson_r"], color=colors_bar)
        ax.axvline(0, color="black", linewidth=0.8)
        # Mark significant
        for i, (_, row) in enumerate(sub.iterrows()):
            if pd.notna(row["pearson_p"]) and row["pearson_p"] < 0.05:
                ax.text(row["pearson_r"], i, " *", va="center", fontsize=8)
        ax.set_title(f"{ds}\nOutcome: {first_out}", fontsize=9, fontweight="bold")
        ax.set_xlabel("Pearson r", fontsize=8)
        ax.tick_params(labelsize=6.5)

    fig.suptitle("Top Emotion–Outcome Correlations by Dataset (* = p<0.05)",
                 fontsize=11, fontweight="bold")
    fig.savefig(OUTPUT_ROOT / "05_top_correlations.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Saved: 05_top_correlations.png")


✓ Saved: 05_top_correlations.png


## Step 9 — Outcome prediction: cross-validated logistic regression

In [40]:
def run_binary_prediction(conv_df, outcomes):
    """5-fold cross-validated logistic regression for binary outcomes."""
    exclude = {ID_COL, "n_turns", "duration_observed", "dataset"} | set(outcomes)
    feat_cols = [c for c in conv_df.columns
                 if c not in exclude and pd.api.types.is_numeric_dtype(conv_df[c])]
    results = []
    for out in outcomes:
        if out not in conv_df.columns:
            continue
        y = pd.to_numeric(conv_df[out], errors="coerce")
        keep = y.isin([0, 1])
        y_ = y[keep].astype(int)
        X_ = conv_df.loc[keep, feat_cols].replace([np.inf,-np.inf], np.nan).fillna(0)
        if len(y_) < 20 or y_.nunique() < 2 or y_.value_counts().min() < 3:
            continue
        folds = min(N_FOLDS, int(y_.value_counts().min()))
        if folds < 2:
            continue
        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(max_iter=5000, class_weight="balanced",
                                          solver="liblinear", random_state=42)),
        ])
        cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
        try:
            y_prob = cross_val_predict(clf, X_, y_, cv=cv, method="predict_proba")[:,1]
            y_pred = (y_prob >= 0.5).astype(int)
            results.append({
                "outcome": out,
                "n": len(y_), "n_pos": int(y_.sum()), "n_neg": int((y_==0).sum()),
                "cv_auc":   round(roc_auc_score(y_, y_prob), 4),
                "accuracy": round(accuracy_score(y_, y_pred), 4),
                "f1":       round(f1_score(y_, y_pred, zero_division=0), 4),
                "precision":round(precision_score(y_, y_pred, zero_division=0), 4),
                "recall":   round(recall_score(y_, y_pred, zero_division=0), 4),
            })
        except Exception as e:
            results.append({"outcome": out, "error": str(e)})
    return pd.DataFrame(results)


all_pred = {}
for ds, conv in all_conv.items():
    outs = all_outcomes[ds]
    if not outs:
        continue
    pred = run_binary_prediction(conv, outs)
    pred["dataset"] = ds
    if not pred.empty:
        pred.to_csv(OUTPUT_ROOT / ds / "binary_prediction.csv", index=False)
        all_pred[ds] = pred
        print(f"  ✓ {ds}:")
        cols = [c for c in ["outcome","n","n_pos","n_neg","cv_auc","f1"]
                if c in pred.columns]
        print(pred[cols].to_string(index=False))

if all_pred:
    pd.concat(all_pred.values(), ignore_index=True).to_csv(
        OUTPUT_ROOT / "06_binary_prediction_summary.csv", index=False)
    print("\n✓ Combined prediction summary saved.")


  ✓ dealerships:
       outcome  n  n_pos  n_neg  cv_auc     f1
outcome_binary 48     23     25  0.6939 0.6512
  ✓ a2a:
       outcome   n  n_pos  n_neg  cv_auc     f1
outcome_binary 405    247    158  0.9702 0.9474
  ✓ emaad:
       outcome   n  n_pos  n_neg  cv_auc     f1
outcome_binary 200    100    100  0.9967 0.9756

✓ Combined prediction summary saved.


In [41]:
if all_pred:
    pred_combined = pd.concat(all_pred.values(), ignore_index=True)
    pred_combined = pred_combined[pred_combined["cv_auc"].notna()]

    if not pred_combined.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        ds_list = list(all_pred.keys())
        x = np.arange(len(ds_list))

        for i, ds in enumerate(ds_list):
            sub = pred_combined[pred_combined["dataset"]==ds]
            for j, (_, row) in enumerate(sub.iterrows()):
                n_ds_outcomes = len(sub)
                w = 0.7 / max(n_ds_outcomes, 1)
                offset = (j - n_ds_outcomes/2 + 0.5) * w
                bar = ax.bar(i + offset, row["cv_auc"], w,
                             color=DS_COLORS.get(ds, f"C{i}"), alpha=0.8)
                ax.text(i + offset, row["cv_auc"] + 0.012,
                        f"{row.get('outcome','')[:8]}\n{row['cv_auc']:.3f}",
                        ha="center", va="bottom", fontsize=6.5, rotation=0)

        ax.axhline(0.5, color="red", linestyle="--", linewidth=0.9, label="Chance (AUC=0.5)")
        ax.set_xticks(x)
        ax.set_xticklabels(ds_list, fontsize=11)
        ax.set_ylabel("CV AUC (5-fold stratified)", fontsize=10)
        ax.set_title("Binary Outcome Prediction AUC Across Datasets\n"
                     "(emotion trajectory features only — no price information)", fontsize=11)
        ax.set_ylim(0, 1.12)
        ax.legend(fontsize=9)
        # Colour legend patches
        import matplotlib.patches as mpatches
        legend_patches = [mpatches.Patch(color=DS_COLORS.get(ds,"#888"), label=ds)
                          for ds in ds_list]
        ax.legend(handles=legend_patches + [
            plt.Line2D([0],[0], color="red", linestyle="--", label="Chance")
        ], fontsize=9)
        plt.tight_layout()
        fig.savefig(OUTPUT_ROOT / "06_auc_comparison.png", dpi=150)
        plt.close()
        print("✓ Saved: 06_auc_comparison.png")


✓ Saved: 06_auc_comparison.png


## Step 10 — Frequency vs predictive importance (scatter)

In [42]:
if all_corr:
    n_cols = len(all_corr)
    fig, axes = plt.subplots(1, n_cols, figsize=(5.5*n_cols, 5.5), constrained_layout=True)
    if n_cols == 1:
        axes = [axes]

    for ax, (ds, corr) in zip(axes, all_corr.items()):
        df = all_data[ds]
        emo = all_emotions[ds]
        means = df[emo].mean()

        # Best |r| per emotion across all outcomes, using _mean features only
        emo_mean_feats = [f"{e}_mean" for e in emo]
        sub = corr[corr["feature"].isin(emo_mean_feats)].copy()
        sub["emotion"] = sub["feature"].str.replace("_mean","",regex=False)
        best_r = sub.groupby("emotion")["abs_pearson_r"].max()

        emotions_plot = [e for e in emo if e in best_r.index and e in means.index]
        x_vals = [float(means[e]) for e in emotions_plot]
        y_vals = [float(best_r[e]) for e in emotions_plot]

        scatter = ax.scatter(x_vals, y_vals,
                             c=DS_COLORS.get(ds,"#888"),
                             alpha=0.75, s=55, edgecolors="white", linewidths=0.5)
        for e, xv, yv in zip(emotions_plot, x_vals, y_vals):
            ax.annotate(e, (xv, yv), fontsize=5.5, xytext=(3,3),
                        textcoords="offset points")

        ax.axhline(0.1, color="gray", linestyle=":", linewidth=0.8, alpha=0.7)
        ax.set_xlabel("Mean activation (frequency proxy)", fontsize=8)
        ax.set_ylabel("Max |Pearson r| with any outcome", fontsize=8)
        ax.set_title(f"{ds}", fontsize=9, fontweight="bold")
        ax.grid(True, alpha=0.25)

    fig.suptitle("Emotion Frequency vs Predictive Importance\n"
                 "(rare emotions can be more predictive than frequent ones)",
                 fontsize=10, fontweight="bold")
    fig.savefig(OUTPUT_ROOT / "07_frequency_vs_importance.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Saved: 07_frequency_vs_importance.png")


✓ Saved: 07_frequency_vs_importance.png


## Final — Master summary table

In [43]:
summary_rows = []
for ds in all_data:
    df   = all_data[ds]
    conv = all_conv.get(ds, pd.DataFrame())
    corr = all_corr.get(ds, pd.DataFrame())
    pred = all_pred.get(ds, pd.DataFrame())

    top_emo   = df[all_emotions[ds]].mean().idxmax()
    anger_rank = (
        int(df[all_emotions[ds]].mean().rank(ascending=False)["anger"])
        if "anger" in all_emotions[ds] else "N/A"
    )
    n_sig_corr = int((corr["pearson_p"] < 0.05).sum()) if not corr.empty and "pearson_p" in corr.columns else 0
    best_auc   = pred["cv_auc"].max() if not pred.empty and "cv_auc" in pred.columns else None

    summary_rows.append({
        "Dataset":             ds,
        "Conversations":       df[ID_COL].nunique(),
        "Total turns":         len(df),
        "#1 emotion":          top_emo,
        "Anger rank":          anger_rank,
        "Sig correlations":    n_sig_corr,
        "Best CV AUC":         round(float(best_auc), 3) if best_auc else "—",
        "Outcomes used":       ", ".join(all_outcomes.get(ds, [])) or "—",
    })

final_df = pd.DataFrame(summary_rows).set_index("Dataset")
final_df.to_csv(OUTPUT_ROOT / "MASTER_SUMMARY.csv")

print("=" * 70)
print("MASTER SUMMARY — ALL DATASETS")
print("=" * 70)
print(final_df.to_string())
print(f"\n✓ All outputs saved to: {OUTPUT_ROOT}")
print("\nOutput files:")
for f in sorted(OUTPUT_ROOT.glob("*.png")) + sorted(OUTPUT_ROOT.glob("*.csv")):
    print(f"  {f.name}")


MASTER SUMMARY — ALL DATASETS
             Conversations  Total turns  #1 emotion  Anger rank  Sig correlations Best CV AUC   Outcomes used
Dataset                                                                                                      
dealerships             48         2480    approval          20                37       0.694  outcome_binary
heddaya                178         6384    approval          21                 0           —               —
casino                1030        14232    approval          23                 0           —               —
a2a                    405         3332  admiration          23               206        0.97  outcome_binary
emaad                  200         3414    approval          20               182       0.997  outcome_binary

✓ All outputs saved to: /content/optimal-nego/outputs/model_free_multi_dataset

Output files:
  01_activation_heatmap.png
  02_cross_dataset_top8_trajectories.png
  03_buyer_seller_asymmetry_summary.

# PART 2 — Dynamic Modeling Pipeline

The cells below are copied from the Stage 1.5 dynamic modeling notebook.  
They intentionally reuse the same output layout and save structure as the original dynamic modeling workflow.


# Stage 1.5 — Formal Dynamic Validation
## Emotional Coupling, Directionality & Predictive Structure in Negotiations

**Purpose:** Supplement the descriptive Stage 1 pipeline with formal dynamic models,
addressing the advisor's core question:

> *"Are these emotions actually structured and predictive, or are we just visualizing smooth random walks?"*

**Coverage:**
- **Section A** — Are emotions structured time series? (AR models vs random walk)
- **Section B** — Who follows whom? (Lag regressions, cross-lag, Granger causality)
- **Section C** — Which emotions predict outcomes? (Ridge / LASSO logistic regression)
- **Section D** — Emotional symmetry: buyer-only, seller-only, difference models

**Prerequisite:** Run the Stage 1 multi-dataset pipeline first.  
This notebook reads the `stage1_turns_embedded.csv` files directly.


In [44]:
!pip install -q numpy pandas scipy scikit-learn matplotlib seaborn statsmodels


## Imports and configuration

In [45]:
from __future__ import annotations
import warnings, os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats
from scipy.stats import pearsonr

from sklearn.linear_model import (
    LogisticRegression, Ridge, Lasso, RidgeCV, LassoCV,
    LinearRegression,
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.metrics import roc_auc_score, r2_score, mean_squared_error
from sklearn.pipeline import Pipeline

import statsmodels.api as sm
from statsmodels.tsa.stattools import acf, pacf, grangercausalitytests, adfuller
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.vector_ar.var_model import VAR

warnings.filterwarnings("ignore")
print("✓ All imports OK")


✓ All imports OK


In [46]:
# ─── EDIT THESE PATHS ────────────────────────────────────────────────────────
# Combined 5-dataset configuration.
# Assumption: Stage 1 pipeline outputs are stored under /content/optimal-nego/outputs/.
# A2A results are stored separately, so multiple likely locations are provided below.

BASE_DIR = Path("/content/optimal-nego")
OUTPUT_DIR = BASE_DIR / "outputs"

def resolve_input_path(*candidates):
    """Return first existing candidate; otherwise return the first candidate for transparent failure."""
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return str(p)
    return str(Path(candidates[0]))

DATASETS = {
    "dealerships": resolve_input_path(
        Path("/content/stage1_turns_dealership-nego.csv"),
        OUTPUT_DIR / "dealerships_nego" / "stage1_turns_embedded.csv",
        Path("/content/optimal-nego/outputs/dealerships/stage1_turns_embedded.csv"),
    ),
    "heddaya": resolve_input_path(
        Path("/content/stage1_turns_heddaya-nego.csv"),
        OUTPUT_DIR / "heddaya_nego" / "stage1_turns_embedded.csv",
        Path("/content/optimal-nego/outputs/heddaya/stage1_turns_embedded.csv"),
    ),
    "casino": resolve_input_path(
        Path("/content/stage1_turns_casino.csv"),
        OUTPUT_DIR / "casino_nego" / "stage1_turns_embedded.csv",
        Path("/content/optimal-nego/outputs/casino/stage1_turns_embedded.csv"),
    ),
    "a2a": resolve_input_path(
        Path("/content/stage1_turns_a2a-nego.csv"),
        OUTPUT_DIR / "a2a_nego" / "stage1_turns_embedded.csv",
        OUTPUT_DIR / "a2a_negotiations" / "stage1_turns_embedded.csv",
        Path("/content/stage1_turns_embedded.csv"),
        Path("/content/a2a_negotiations/stage1_turns_embedded.csv"),
        Path("/mnt/data/stage1_turns_embedded.csv"),
    ),
    "emaad": resolve_input_path(
        Path("/content/stage1_turns_emaad-sales.csv"),
        OUTPUT_DIR / "emaad_sales" / "stage1_turns_embedded.csv",
        Path("/content/optimal-nego/outputs/emaad/stage1_turns_embedded.csv"),
    ),
}

OUTPUT_ROOT = Path("/content/optimal-nego/outputs/stage1_5_dynamic")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Analysis settings
N_LAGS   = 3        # lags to test in AR and cross-lag models
N_BOOT   = 500      # bootstrap iterations for CIs
N_FOLDS  = 5        # CV folds for outcome models
N_BINS   = 10       # time bins for trajectory plots
MIN_OBS  = 10       # minimum turns per conversation to include

# Visual settings
DS_COLORS = {
    "dealerships": "#E05A47",
    "heddaya":     "#1A8A7A",
    "casino":      "#7B68C8",
    "a2a":         "#3B82F6",
    "emaad":       "#E8A945",
}
ROLE_COLORS = {"buyer": "#4C9EEB", "seller": "#F0956A"}

# Columns
ID_COL   = "conversation_id"
TURN_COL = "turn_index"
ROLE_COL = "role"

GOEMOTIONS_27 = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval","disgust",
    "embarrassment","excitement","fear","gratitude","grief","joy","love",
    "nervousness","optimism","pride","realization","relief","remorse",
    "sadness","surprise",
]

# Focal emotions for detailed analysis (covers all theoretically important ones)
FOCAL_EMOTIONS = [
    "approval","curiosity","admiration","gratitude","remorse",
    "anger","disapproval","disappointment","annoyance","fear",
]

print(f"✓ Config ready. Output: {OUTPUT_ROOT}")

✓ Config ready. Output: /content/optimal-nego/outputs/stage1_5_dynamic


## Data loading

In [47]:
def load_and_prep(path: str, dataset_name: str) -> pd.DataFrame:
    """Load Stage 1 CSV, standardise roles, infer binary outcome."""
    df = pd.read_csv(path)
    df = df.sort_values([ID_COL, TURN_COL]).reset_index(drop=True)
    df[ROLE_COL] = df[ROLE_COL].astype(str).str.lower().str.strip()
    df["dataset"] = dataset_name

    # Binary outcome
    if "outcome_binary" not in df.columns:
        if "outcome" in df.columns:
            s = df["outcome"].astype(str).str.lower().str.strip()
            df["outcome_binary"] = np.nan
            df.loc[s.isin(["sale","deal","success","sold","yes","1"]),"outcome_binary"] = 1
            df.loc[s.isin(["no sale","no deal","failure","failed","no","0"]),"outcome_binary"] = 0

    # Relative turn position [0,1]
    def norm(x):
        d = max(1, x.max() - x.min())
        return (x - x.min()) / d
    df["relative_time"] = df.groupby(ID_COL)[TURN_COL].transform(norm)
    return df


def detect_emotions(df: pd.DataFrame) -> List[str]:
    known = [c for c in GOEMOTIONS_27
             if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    return known if known else [
        c for c in df.columns
        if c not in {ID_COL,TURN_COL,ROLE_COL,"outcome","outcome_binary",
                     "relative_time","dataset","text","start_time","end_time",
                     "speaker_id","global_row_id"}
        and pd.api.types.is_numeric_dtype(df[c])
    ]


all_data     = {}
all_emotions = {}

for ds, path in DATASETS.items():
    if not Path(path).exists():
        print(f"  ✗ {ds}: not found")
        continue
    df  = load_and_prep(path, ds)
    emo = detect_emotions(df)
    all_data[ds]     = df
    all_emotions[ds] = emo
    has_outcome = df["outcome_binary"].notna().any() if "outcome_binary" in df.columns else False
    print(f"  ✓ {ds:15s} | {df[ID_COL].nunique():4d} convs "
          f"| {len(df):6d} turns | {len(emo)} dims | outcome={has_outcome}")

print(f"\n✓ Loaded {len(all_data)} datasets")


  ✓ dealerships     |   48 convs |   2480 turns | 27 dims | outcome=True
  ✓ heddaya         |  178 convs |   6384 turns | 27 dims | outcome=False
  ✓ casino          | 1030 convs |  14232 turns | 27 dims | outcome=True
  ✓ a2a             |  405 convs |   3332 turns | 27 dims | outcome=True
  ✓ emaad           |  200 convs |   3414 turns | 27 dims | outcome=True

✓ Loaded 5 datasets


---
## Section A — Are emotions structured time series?
### Core question: random walk or structured temporal process?

For each emotion, we test:
1. **AR(1/2/3) models** — does past emotion predict current emotion better than chance?
2. **Random walk benchmark** — does E_t = E_{t-1} + ε fit as well as AR(p)?
3. **ADF test** — is the series stationary (structured) or a unit-root process (random walk)?
4. **ACF/PACF plots** — visualise the lag structure for focal emotions.

*Logic: If emotions were random walks, AR models would show no predictive advantage over the benchmark and ADF tests would fail to reject a unit root.*


In [48]:
def build_turn_sequences(df: pd.DataFrame, emotion: str,
                          role: Optional[str] = None) -> List[np.ndarray]:
    """
    Extract per-conversation turn sequences for one emotion.
    Optionally filter by role. Returns list of 1-D arrays.
    Logic: We model within-conversation temporal dynamics.
    Cross-conversation averaging would destroy the sequential structure.
    """
    sub = df.copy()
    if role:
        sub = sub[sub[ROLE_COL] == role]
    sequences = []
    for cid, g in sub.groupby(ID_COL):
        g = g.sort_values(TURN_COL)
        vals = g[emotion].dropna().to_numpy(dtype=float)
        if len(vals) >= MIN_OBS:
            sequences.append(vals)
    return sequences


def build_lag_dataframe(sequences: List[np.ndarray],
                         n_lags: int = N_LAGS) -> pd.DataFrame:
    """
    Stack all conversations into a single lag design matrix.
    Logic: Pooled regression across conversations.
    Conversations are independent, so pooling is valid and increases power.
    First n_lags rows of each conversation are dropped (no prior turns).
    """
    rows = []
    for seq in sequences:
        if len(seq) <= n_lags:
            continue
        for t in range(n_lags, len(seq)):
            row = {"y": seq[t]}
            for lag in range(1, n_lags + 1):
                row[f"lag{lag}"] = seq[t - lag]
            rows.append(row)
    return pd.DataFrame(rows).dropna()


def fit_ar_models(df_lag: pd.DataFrame) -> Dict:
    """
    Fit AR(1), AR(2), AR(3) and random walk benchmark.
    Returns dict of {model_name: {aic, bic, r2_in, rmse_oos}}.
    Logic: OLS for speed and interpretability. We report in-sample R2
    (how much variance is explained by lagged emotions) and out-of-sample
    RMSE on held-out 20% to detect overfitting.
    """
    if len(df_lag) < 30:
        return {}

    y = df_lag["y"].values
    results = {}

    for name, feat_cols in [
        ("RW_benchmark", ["lag1"]),    # Random walk: y_t ≈ y_{t-1}
        ("AR1",          ["lag1"]),     # AR(1)
        ("AR2",          ["lag1","lag2"]),
        ("AR3",          ["lag1","lag2","lag3"]),
    ]:
        if not all(c in df_lag.columns for c in feat_cols):
            continue
        X = df_lag[feat_cols].values

        # Train / test split (last 20%)
        split = int(len(y) * 0.8)
        X_tr, X_te = X[:split], X[split:]
        y_tr, y_te = y[:split], y[split:]
        if len(y_te) < 5:
            continue

        if name == "RW_benchmark":
            # RW prediction: y_hat = y_{t-1} (no fit, just persistence)
            y_pred_oos = X_te[:, 0]
            residuals  = y_tr - X_tr[:, 0]
            ss_tot = np.var(y_tr) * len(y_tr)
            ss_res = np.sum(residuals**2)
            r2_in  = max(0, 1 - ss_res / ss_tot) if ss_tot > 0 else 0
            rmse   = float(np.sqrt(np.mean((y_te - y_pred_oos)**2)))
            # AIC/BIC: 1 parameter (implicit), k=1
            n, k = len(y_tr), 1
            sse  = float(np.sum(residuals**2))
            aic  = n * np.log(sse/n) + 2*k
            bic  = n * np.log(sse/n) + k * np.log(n)
        else:
            Xc_tr = sm.add_constant(X_tr)
            Xc_te = sm.add_constant(X_te)
            try:
                ols = sm.OLS(y_tr, Xc_tr).fit()
            except Exception:
                continue
            y_pred_oos = ols.predict(Xc_te)
            r2_in  = float(max(0, ols.rsquared))
            rmse   = float(np.sqrt(mean_squared_error(y_te, y_pred_oos)))
            aic    = float(ols.aic)
            bic    = float(ols.bic)

        results[name] = {
            "aic": round(aic, 2), "bic": round(bic, 2),
            "r2_in": round(r2_in, 4), "rmse_oos": round(rmse, 6),
        }

    return results


def adf_test(sequences: List[np.ndarray]) -> Dict:
    """
    ADF test for unit root pooled across conversations.
    Logic: A unit root = random walk. If we can reject it (p < 0.05),
    the series is stationary and AR models are valid.
    We pool all conversation sequences and run one pooled ADF.
    """
    pooled = np.concatenate(sequences)
    if len(pooled) < 20:
        return {"adf_stat": np.nan, "p_value": np.nan, "stationary": np.nan}
    try:
        result = adfuller(pooled, autolag="AIC")
        return {
            "adf_stat":   round(float(result[0]), 4),
            "p_value":    round(float(result[1]), 4),
            "stationary": result[1] < 0.05,
        }
    except Exception:
        return {"adf_stat": np.nan, "p_value": np.nan, "stationary": np.nan}


print("Section A functions defined. Running AR analysis across all datasets...")

section_a_results = {}

for ds, df in all_data.items():
    ds_dir = OUTPUT_ROOT / ds
    ds_dir.mkdir(exist_ok=True)
    emo_list = all_emotions[ds]
    rows     = []

    for emo in emo_list:
        seqs    = build_turn_sequences(df, emo)
        df_lag  = build_lag_dataframe(seqs)
        ar_res  = fit_ar_models(df_lag)
        adf_res = adf_test(seqs)

        if not ar_res:
            continue

        rw_rmse  = ar_res.get("RW_benchmark", {}).get("rmse_oos", np.nan)
        ar1_rmse = ar_res.get("AR1",          {}).get("rmse_oos", np.nan)
        ar3_r2   = ar_res.get("AR3",          {}).get("r2_in",    np.nan)
        improvement = (
            round((rw_rmse - ar1_rmse) / rw_rmse * 100, 2)
            if pd.notna(rw_rmse) and pd.notna(ar1_rmse) and rw_rmse > 0 else np.nan
        )

        rows.append({
            "emotion":          emo,
            "adf_stat":         adf_res["adf_stat"],
            "adf_p":            adf_res["p_value"],
            "stationary":       adf_res["stationary"],
            "rw_rmse":          rw_rmse,
            "ar1_r2_in":        ar_res.get("AR1",{}).get("r2_in", np.nan),
            "ar1_rmse_oos":     ar1_rmse,
            "ar3_r2_in":        ar3_r2,
            "ar3_aic":          ar_res.get("AR3",{}).get("aic", np.nan),
            "ar1_beats_rw_pct": improvement,
            "structured":       improvement is not np.nan and improvement > 0,
        })

    result_df = pd.DataFrame(rows).sort_values("ar1_beats_rw_pct", ascending=False)
    result_df.to_csv(ds_dir / "A_ar_vs_random_walk.csv", index=False)
    section_a_results[ds] = result_df

    n_stat    = result_df["stationary"].sum()
    n_struct  = result_df["structured"].sum()
    mean_r2   = result_df["ar1_r2_in"].mean()
    print(f"  {ds}: {n_stat}/{len(result_df)} stationary | "
          f"{n_struct}/{len(result_df)} AR beats RW | mean AR1 R²={mean_r2:.3f}")


Section A functions defined. Running AR analysis across all datasets...
  dealerships: 27/27 stationary | 27/27 AR beats RW | mean AR1 R²=0.003
  heddaya: 27/27 stationary | 27/27 AR beats RW | mean AR1 R²=0.004
  casino: 27/27 stationary | 27/27 AR beats RW | mean AR1 R²=0.006
  a2a: 27/27 stationary | 14/27 AR beats RW | mean AR1 R²=0.030
  emaad: 27/27 stationary | 27/27 AR beats RW | mean AR1 R²=0.008


In [49]:
# ─── Plot: AR1 R² and % improvement over random walk ───────────────────────
fig, axes = plt.subplots(1, len(section_a_results), figsize=(6*len(section_a_results), 7),
                          constrained_layout=True)
if len(section_a_results) == 1:
    axes = [axes]

for ax, (ds, res) in zip(axes, section_a_results.items()):
    res_plot = res.sort_values("ar1_r2_in", ascending=True).tail(20)
    colors   = [DS_COLORS.get(ds,"#888") if r else "#CCCCCC"
                for r in res_plot["structured"]]
    ax.barh(res_plot["emotion"], res_plot["ar1_r2_in"], color=colors, alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("AR(1) in-sample R²", fontsize=9)
    ax.set_title(f"{ds}\nAR(1) R² per emotion\n(colored = beats random walk)",
                 fontsize=9, fontweight="bold")
    ax.tick_params(labelsize=7)

fig.suptitle("Section A — Emotional Predictability: AR(1) vs Random Walk\n"
             "(R² > 0 = structured; R² ≈ 0 = noise)", fontsize=11, fontweight="bold")
fig.savefig(OUTPUT_ROOT / "A1_ar_r2_by_emotion.png", dpi=150, bbox_inches="tight")
plt.close()

# ─── Cross-dataset summary table ───────────────────────────────────────────
summary_a = pd.concat(
    [r.assign(dataset=ds)[["dataset","emotion","ar1_r2_in","ar1_beats_rw_pct","stationary","adf_p"]]
     for ds, r in section_a_results.items()],
    ignore_index=True
)
pivot_r2 = summary_a.pivot(index="emotion", columns="dataset", values="ar1_r2_in")
pivot_r2.to_csv(OUTPUT_ROOT / "A_cross_dataset_ar1_r2.csv")

print("Cross-dataset AR(1) R² pivot (higher = more structured):")
print(pivot_r2.round(4).fillna("—").to_string())
print("\n✓ Section A plots saved.")


Cross-dataset AR(1) R² pivot (higher = more structured):
dataset            a2a  casino  dealerships   emaad  heddaya
emotion                                                     
admiration      0.0009  0.0067       0.0082  0.0058   0.0111
amusement       0.1366  0.0084       0.0000  0.0000   0.0000
anger           0.0005  0.0004       0.0008  0.0000   0.0000
annoyance       0.0575  0.0078       0.0069  0.0000   0.0009
approval        0.0047  0.0040       0.0001  0.0040   0.0088
caring          0.0670  0.0049       0.0002  0.0000   0.0001
confusion       0.0009  0.0004       0.0048  0.0248   0.0031
curiosity       0.0057  0.0009       0.0033  0.1009   0.0000
desire          0.0001  0.0087       0.0004  0.0001   0.0032
disappointment  0.0175  0.0030       0.0005  0.0001   0.0020
disapproval     0.0467  0.0023       0.0008  0.0007   0.0004
disgust         0.0049  0.0001       0.0000  0.0119   0.0021
embarrassment   0.0112  0.0002       0.0000  0.0028   0.0008
excitement      0.0032  0.01

In [50]:
# ─── ACF / PACF plots for focal emotions (one dataset at a time) ─────────────
# Shows lag structure — how many lags carry signal

ds_for_acf = [ds for ds in ["heddaya","dealerships","casino","emaad"] if ds in all_data][0]
df_acf = all_data[ds_for_acf]
emo_acf = [e for e in FOCAL_EMOTIONS if e in all_emotions[ds_for_acf]][:6]

fig, axes = plt.subplots(len(emo_acf), 2, figsize=(10, len(emo_acf)*2.5),
                          constrained_layout=True)

for row_i, emo in enumerate(emo_acf):
    seqs   = build_turn_sequences(df_acf, emo)
    pooled = np.concatenate(seqs) if seqs else np.array([])
    if len(pooled) < 20:
        continue

    ax_acf  = axes[row_i, 0]
    ax_pacf = axes[row_i, 1]

    try:
        acf_vals  = acf(pooled,  nlags=N_LAGS*4, fft=True)
        pacf_vals = pacf(pooled, nlags=N_LAGS*4, method="ols")
        conf = 1.96 / np.sqrt(len(pooled))

        ax_acf.bar(range(len(acf_vals)), acf_vals, color=DS_COLORS.get(ds_for_acf,"#888"), alpha=0.7)
        ax_acf.axhline(conf,  color="red", linestyle="--", linewidth=0.8)
        ax_acf.axhline(-conf, color="red", linestyle="--", linewidth=0.8)
        ax_acf.axhline(0, color="black", linewidth=0.5)
        ax_acf.set_title(f"{emo} — ACF", fontsize=8)
        ax_acf.set_xlabel("Lag", fontsize=7)
        ax_acf.tick_params(labelsize=6)

        ax_pacf.bar(range(len(pacf_vals)), pacf_vals, color=ROLE_COLORS["buyer"], alpha=0.7)
        ax_pacf.axhline(conf,  color="red", linestyle="--", linewidth=0.8)
        ax_pacf.axhline(-conf, color="red", linestyle="--", linewidth=0.8)
        ax_pacf.axhline(0, color="black", linewidth=0.5)
        ax_pacf.set_title(f"{emo} — PACF", fontsize=8)
        ax_pacf.set_xlabel("Lag", fontsize=7)
        ax_pacf.tick_params(labelsize=6)
    except Exception as e:
        ax_acf.set_title(f"{emo} — error: {e}", fontsize=7)

fig.suptitle(f"ACF and PACF for Focal Emotions — {ds_for_acf}\n"
             "(Red dashed = 95% significance threshold; bars outside = structured lag)",
             fontsize=10, fontweight="bold")
fig.savefig(OUTPUT_ROOT / f"A2_acf_pacf_{ds_for_acf}.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"✓ Saved: A2_acf_pacf_{ds_for_acf}.png")


✓ Saved: A2_acf_pacf_heddaya.png


---
## Section B — Who follows whom? Directional emotional influence

**Core question:** Does buyer emotion at t−1 predict seller emotion at t, or vice versa?

We run three complementary tests:
1. **Cross-lag OLS** — regress seller_t on buyer_{t-1} (and reverse)
2. **Granger causality test** — formal test of whether one series improves prediction of the other
3. **Cross-Correlation Function (CCF)** — visual map of lead-lag structure at all lags

*Logic: If β(buyer → seller) > β(seller → buyer), the buyer is the emotional leader.
Granger causality formalises this asymmetry statistically.*


In [51]:
def build_cross_lag_df(df: pd.DataFrame, emotion: str,
                        n_lags: int = N_LAGS) -> Optional[pd.DataFrame]:
    """
    Build paired buyer-seller cross-lag design matrix.
    For each conversation, align buyer and seller turns by position.
    Logic: We pivot to wide format (one row per turn position) so we can
    regress seller_t on buyer_{t-1} within the same conversation.
    This is the standard cross-lagged panel model setup.
    """
    rows = []
    for cid, g in df.groupby(ID_COL):
        g      = g.sort_values(TURN_COL)
        buyer  = g[g[ROLE_COL]=="buyer" ][["relative_time", emotion]].reset_index(drop=True)
        seller = g[g[ROLE_COL]=="seller"][["relative_time", emotion]].reset_index(drop=True)

        # Need at least n_lags+1 turns from each role
        min_len = min(len(buyer), len(seller))
        if min_len < n_lags + 3:
            continue

        buyer_vals  = buyer[emotion].values[:min_len]
        seller_vals = seller[emotion].values[:min_len]

        for t in range(n_lags, min_len):
            row = {
                "buyer_t":   buyer_vals[t],
                "seller_t":  seller_vals[t],
                "conv_id":   cid,
            }
            for lag in range(1, n_lags + 1):
                row[f"buyer_lag{lag}"]  = buyer_vals[t - lag]
                row[f"seller_lag{lag}"] = seller_vals[t - lag]
            rows.append(row)

    if not rows:
        return None
    return pd.DataFrame(rows).dropna()


def fit_cross_lag_models(cross_df: pd.DataFrame,
                          n_lags: int = N_LAGS) -> Dict:
    """
    Fit bidirectional cross-lag regressions.
    Model A: seller_t ~ buyer_lag1 ... buyer_lag_n  (buyer → seller)
    Model B: buyer_t  ~ seller_lag1 ... seller_lag_n (seller → buyer)
    Returns coefficients, R², and direction test.
    """
    if cross_df is None or len(cross_df) < 30:
        return {}

    results = {}
    for direction, y_col, lag_prefix in [
        ("buyer_to_seller", "seller_t", "buyer_lag"),
        ("seller_to_buyer", "buyer_t",  "seller_lag"),
    ]:
        lag_cols = [f"{lag_prefix}{i}" for i in range(1, n_lags+1)
                    if f"{lag_prefix}{i}" in cross_df.columns]
        if not lag_cols:
            continue

        y = cross_df[y_col].values
        X = sm.add_constant(cross_df[lag_cols].values)

        try:
            ols  = sm.OLS(y, X).fit()
            coefs = {lag_cols[i]: round(float(ols.params[i+1]), 5)
                     for i in range(len(lag_cols))}
            pvals = {lag_cols[i]: round(float(ols.pvalues[i+1]), 4)
                     for i in range(len(lag_cols))}
            results[direction] = {
                "r2":        round(float(ols.rsquared), 4),
                "aic":       round(float(ols.aic), 2),
                "n":         len(y),
                "coefs":     coefs,
                "pvalues":   pvals,
                "lag1_coef": round(float(ols.params[1]), 5),
                "lag1_p":    round(float(ols.pvalues[1]), 4),
                "any_sig":   any(p < 0.05 for p in ols.pvalues[1:]),
            }
        except Exception:
            continue

    return results


def granger_test(sequences_a: List[np.ndarray],
                  sequences_b: List[np.ndarray],
                  n_lags: int = N_LAGS) -> Dict:
    """
    Pooled Granger causality: does A Granger-cause B?
    Logic: Granger causality tests whether including lagged A improves
    prediction of B beyond B's own lags alone. We pool conversations.
    Returns min p-value across lag specifications.
    """
    a_pool = np.concatenate(sequences_a) if sequences_a else np.array([])
    b_pool = np.concatenate(sequences_b) if sequences_b else np.array([])
    min_len = min(len(a_pool), len(b_pool))
    if min_len < 50:
        return {"min_p": np.nan, "granger_sig": np.nan}
    try:
        data   = np.column_stack([b_pool[:min_len], a_pool[:min_len]])
        gc_res = grangercausalitytests(data, maxlag=n_lags, verbose=False)
        min_p  = min(gc_res[lag][0]["ssr_ftest"][1] for lag in gc_res)
        return {"min_p": round(min_p, 4), "granger_sig": min_p < 0.05}
    except Exception:
        return {"min_p": np.nan, "granger_sig": np.nan}


print("Section B: Running cross-lag and Granger causality analysis...")

section_b_results = {}

for ds, df in all_data.items():
    ds_dir  = OUTPUT_ROOT / ds
    ds_dir.mkdir(exist_ok=True)
    emo_list = all_emotions[ds]
    rows     = []

    for emo in emo_list:
        cross_df = build_cross_lag_df(df, emo)
        cl_res   = fit_cross_lag_models(cross_df) if cross_df is not None else {}

        # Granger tests
        buyer_seqs  = build_turn_sequences(df, emo, role="buyer")
        seller_seqs = build_turn_sequences(df, emo, role="seller")
        gc_b2s = granger_test(buyer_seqs,  seller_seqs)  # buyer → seller
        gc_s2b = granger_test(seller_seqs, buyer_seqs)   # seller → buyer

        b2s = cl_res.get("buyer_to_seller", {})
        s2b = cl_res.get("seller_to_buyer", {})

        row = {
            "emotion":          emo,
            # Cross-lag R²
            "b2s_r2":           b2s.get("r2", np.nan),
            "s2b_r2":           s2b.get("r2", np.nan),
            # Lag-1 coefficients
            "b2s_lag1_coef":    b2s.get("lag1_coef", np.nan),
            "s2b_lag1_coef":    s2b.get("lag1_coef", np.nan),
            "b2s_lag1_p":       b2s.get("lag1_p", np.nan),
            "s2b_lag1_p":       s2b.get("lag1_p", np.nan),
            # Granger p-values
            "gc_buyer2seller_p":  gc_b2s.get("min_p", np.nan),
            "gc_seller2buyer_p":  gc_s2b.get("min_p", np.nan),
            "gc_buyer2seller_sig":gc_b2s.get("granger_sig", False),
            "gc_seller2buyer_sig":gc_s2b.get("granger_sig", False),
        }

        # Direction assessment
        if pd.notna(row["b2s_r2"]) and pd.notna(row["s2b_r2"]):
            if row["b2s_r2"] > row["s2b_r2"] + 0.005:
                row["leader"] = "buyer"
            elif row["s2b_r2"] > row["b2s_r2"] + 0.005:
                row["leader"] = "seller"
            else:
                row["leader"] = "symmetric"
        else:
            row["leader"] = "insufficient_data"

        rows.append(row)

    res_df = pd.DataFrame(rows)
    res_df.to_csv(ds_dir / "B_directional_influence.csv", index=False)
    section_b_results[ds] = res_df

    n_buyer_leads  = (res_df["leader"]=="buyer").sum()
    n_seller_leads = (res_df["leader"]=="seller").sum()
    n_sym          = (res_df["leader"]=="symmetric").sum()
    print(f"  {ds}: buyer leads={n_buyer_leads} | seller leads={n_seller_leads} | symmetric={n_sym}")


Section B: Running cross-lag and Granger causality analysis...
  dealerships: buyer leads=3 | seller leads=5 | symmetric=19
  heddaya: buyer leads=1 | seller leads=3 | symmetric=23
  casino: buyer leads=5 | seller leads=0 | symmetric=22
  a2a: buyer leads=13 | seller leads=8 | symmetric=6
  emaad: buyer leads=8 | seller leads=3 | symmetric=16


In [52]:
# ─── Cross-lag R² comparison plot (buyer→seller vs seller→buyer) ─────────────
fig, axes = plt.subplots(1, len(section_b_results), figsize=(7*len(section_b_results), 7),
                          constrained_layout=True)
if len(section_b_results) == 1:
    axes = [axes]

for ax, (ds, res) in zip(axes, section_b_results.items()):
    valid = res.dropna(subset=["b2s_r2","s2b_r2"]).copy()
    if valid.empty:
        continue
    valid["delta_r2"] = valid["b2s_r2"] - valid["s2b_r2"]
    valid = valid.sort_values("delta_r2")

    colors = []
    for _, row in valid.iterrows():
        if   row["leader"] == "buyer":    colors.append("#4C9EEB")
        elif row["leader"] == "seller":   colors.append("#F0956A")
        else:                             colors.append("#CCCCCC")

    ax.barh(valid["emotion"], valid["delta_r2"], color=colors, alpha=0.85)
    ax.axvline(0, color="black", linewidth=1.0)
    ax.set_xlabel("Δ R²  (buyer→seller  minus  seller→buyer)", fontsize=9)
    ax.set_title(f"{ds}\nDirectional influence (blue=buyer leads, orange=seller leads)",
                 fontsize=9, fontweight="bold")
    ax.tick_params(labelsize=7)

    # Legend
    blue_p  = mpatches.Patch(color="#4C9EEB",  label="Buyer leads")
    org_p   = mpatches.Patch(color="#F0956A",  label="Seller leads")
    grey_p  = mpatches.Patch(color="#CCCCCC",  label="Symmetric")
    ax.legend(handles=[blue_p, org_p, grey_p], fontsize=7, loc="lower right")

fig.suptitle("Section B — Emotional Leadership: ΔR² (Buyer→Seller) − (Seller→Buyer)\n"
             "Positive = buyer leads; Negative = seller leads; Near zero = symmetric",
             fontsize=10, fontweight="bold")
fig.savefig(OUTPUT_ROOT / "B1_directional_delta_r2.png", dpi=150, bbox_inches="tight")
plt.close()
print("✓ Saved: B1_directional_delta_r2.png")


✓ Saved: B1_directional_delta_r2.png


In [53]:
# ─── Cross-Correlation Function (CCF) for focal emotions ─────────────────────
# Shows buyer-seller correlation at all lags — who anticipates whom

ds_ccf  = [d for d in ["heddaya","dealerships","casino","emaad"] if d in all_data][0]
df_ccf  = all_data[ds_ccf]
emo_ccf = [e for e in FOCAL_EMOTIONS if e in all_emotions[ds_ccf]][:6]
max_lag = 6

fig, axes = plt.subplots(len(emo_ccf), 1, figsize=(10, len(emo_ccf)*2.8),
                          constrained_layout=True)
if len(emo_ccf) == 1:
    axes = [axes]

for ax, emo in zip(axes, emo_ccf):
    buyer_seqs  = build_turn_sequences(df_ccf, emo, role="buyer")
    seller_seqs = build_turn_sequences(df_ccf, emo, role="seller")
    if not buyer_seqs or not seller_seqs:
        continue

    # Pool and align to min length
    min_len    = min(sum(len(s) for s in buyer_seqs),
                     sum(len(s) for s in seller_seqs))
    b_pool = np.concatenate(buyer_seqs)[:min_len]
    s_pool = np.concatenate(seller_seqs)[:min_len]

    # Compute cross-correlations at lags -max_lag .. +max_lag
    ccf_vals = []
    for lag in range(-max_lag, max_lag + 1):
        if lag >= 0:
            b_s, s_s = b_pool[:len(b_pool)-lag] if lag > 0 else b_pool,                        s_pool[lag:] if lag > 0 else s_pool
        else:
            b_s = b_pool[-lag:]
            s_s = s_pool[:len(s_pool)+lag]
        if len(b_s) < 10 or len(s_s) != len(b_s):
            ccf_vals.append(np.nan)
            continue
        try:
            r, _ = pearsonr(b_s, s_s)
            ccf_vals.append(r)
        except Exception:
            ccf_vals.append(np.nan)

    lags = range(-max_lag, max_lag + 1)
    conf = 1.96 / np.sqrt(min_len)

    bars = ax.bar(list(lags), ccf_vals, color=[
        "#4C9EEB" if l < 0 else "#F0956A" if l > 0 else "#1A8A7A"
        for l in lags
    ], alpha=0.8)
    ax.axhline(conf,  color="red", linestyle="--", linewidth=0.8, label="95% CI bound")
    ax.axhline(-conf, color="red", linestyle="--", linewidth=0.8)
    ax.axhline(0,     color="black", linewidth=0.5)
    ax.axvline(0,     color="black", linewidth=0.8, alpha=0.4)
    ax.set_title(f"{emo}  (blue=buyer leads seller | orange=seller leads buyer | green=simultaneous)",
                 fontsize=8)
    ax.set_xlabel("Lag (negative=buyer earlier, positive=seller earlier)", fontsize=7)
    ax.set_ylabel("Cross-corr r", fontsize=7)
    ax.tick_params(labelsize=6.5)
    ax.set_xlim(-max_lag-0.5, max_lag+0.5)
    ax.legend(fontsize=6.5)

fig.suptitle(f"Section B — Cross-Correlation Function (CCF): Buyer ↔ Seller\n{ds_ccf}\n"
             "Peak at negative lag = buyer anticipates seller; "
             "Peak at positive lag = seller anticipates buyer",
             fontsize=9, fontweight="bold")
fig.savefig(OUTPUT_ROOT / f"B2_ccf_{ds_ccf}.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"✓ Saved: B2_ccf_{ds_ccf}.png")


✓ Saved: B2_ccf_heddaya.png


In [54]:
# ─── Granger causality heatmap ────────────────────────────────────────────────
# Shows which emotions have significant directional influence in each dataset

fig, axes = plt.subplots(1, len(section_b_results), figsize=(7*len(section_b_results), 9),
                          constrained_layout=True)
if len(section_b_results) == 1:
    axes = [axes]

for ax, (ds, res) in zip(axes, section_b_results.items()):
    valid = res.dropna(subset=["gc_buyer2seller_p","gc_seller2buyer_p"]).copy()
    if valid.empty:
        ax.set_visible(False)
        continue

    # Build heatmap data: rows=emotions, cols=[b→s, s→b]
    heat = valid.set_index("emotion")[["gc_buyer2seller_p","gc_seller2buyer_p"]].copy()
    heat.columns = ["Buyer → Seller", "Seller → Buyer"]
    heat = heat.sort_values("Buyer → Seller")

    sns.heatmap(heat, ax=ax, cmap="RdYlGn_r", vmin=0, vmax=0.1,
                annot=True, fmt=".3f", linewidths=0.4, linecolor="white",
                cbar_kws={"label": "Granger p-value"})
    ax.set_title(f"{ds}\nGranger Causality p-values\n(green < 0.05 = significant influence)",
                 fontsize=9, fontweight="bold")
    ax.tick_params(labelsize=7)

fig.suptitle("Section B — Granger Causality: Directional Emotional Influence\n"
             "Green cells = statistically significant lead-lag influence (p<0.05)",
             fontsize=10, fontweight="bold")
fig.savefig(OUTPUT_ROOT / "B3_granger_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("✓ Saved: B3_granger_heatmap.png")


✓ Saved: B3_granger_heatmap.png


---
## Section C — Which emotions predict outcomes?
### Ridge and LASSO logistic regression with lagged emotion features

**Core question:** Which specific emotions, in which role, at which lag, predict deal success?

We build a rich feature matrix including:
- Mean, slope, early/late means for each emotion × role
- Lagged emotion values at turn level
- Buyer-seller difference features

Then fit:
1. **LASSO** — automatic feature selection (sparse solution)
2. **Ridge** — stabilised coefficients under multicollinearity
3. **Logistic (unregularised)** — baseline comparison


In [55]:
def build_outcome_features(df: pd.DataFrame, emotion_cols: List[str]) -> pd.DataFrame:
    """
    Build conversation-level feature matrix for outcome prediction.
    Logic: We compute summary statistics for each emotion × role combination:
    mean, std, early mean (first third), late mean (last third), delta (late-early),
    and the buyer-seller difference in mean activation.
    These are interpretable and directly address the advisor's request for
    'left hand side = outcome, right side = emotions'.
    """
    rows = []
    for cid, g in df.groupby(ID_COL):
        g    = g.sort_values(TURN_COL)
        row  = {ID_COL: cid}

        # Outcome (constant within conversation)
        if "outcome_binary" in g.columns:
            vals = g["outcome_binary"].dropna()
            row["outcome_binary"] = vals.iloc[0] if len(vals) else np.nan

        row["n_turns"] = len(g)

        for emo in emotion_cols:
            for role, tag in [("buyer","b"), ("seller","s"), (None,"all")]:
                sub = g[g[ROLE_COL]==role] if role else g
                vals = sub[emo].dropna().to_numpy(dtype=float)
                if len(vals) == 0:
                    for sfx in ["mean","std","early","late","delta"]:
                        row[f"{emo}_{tag}_{sfx}"] = np.nan
                    continue
                row[f"{emo}_{tag}_mean"]  = np.mean(vals)
                row[f"{emo}_{tag}_std"]   = np.std(vals)
                row[f"{emo}_{tag}_early"] = np.mean(vals[:max(1,len(vals)//3)])
                row[f"{emo}_{tag}_late"]  = np.mean(vals[-(max(1,len(vals)//3)):])
                row[f"{emo}_{tag}_delta"] = (np.mean(vals[-(max(1,len(vals)//3)):])
                                              - np.mean(vals[:max(1,len(vals)//3)]))

            # Buyer-seller difference
            b = g[g[ROLE_COL]=="buyer"][emo].dropna()
            s = g[g[ROLE_COL]=="seller"][emo].dropna()
            row[f"{emo}_diff_b_minus_s"] = (b.mean() - s.mean()
                                             if len(b) and len(s) else np.nan)
        rows.append(row)

    return pd.DataFrame(rows)


def run_regularised_models(feat_df: pd.DataFrame,
                            outcome_col: str = "outcome_binary") -> Optional[pd.DataFrame]:
    """
    Fit LASSO, Ridge, and standard Logistic on conversation-level features.
    Returns coefficient comparison table.
    Logic: All three models share the same design matrix.
    LASSO selects features (many coefs = 0). Ridge shrinks but retains all.
    Comparison shows which emotions are robustly important vs noise-selected.
    """
    valid = feat_df.dropna(subset=[outcome_col]).copy()
    y_raw = pd.to_numeric(valid[outcome_col], errors="coerce")
    keep  = y_raw.isin([0, 1])
    valid = valid.loc[keep].copy()
    y     = y_raw.loc[keep].astype(int)

    if len(valid) < 20 or y.nunique() < 2 or y.value_counts().min() < 3:
        return None

    feat_cols = [c for c in valid.columns
                 if c not in {ID_COL, outcome_col, "n_turns"}
                 and pd.api.types.is_numeric_dtype(valid[c])]
    X = valid[feat_cols].replace([np.inf,-np.inf], np.nan).fillna(0)

    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X)

    folds = min(N_FOLDS, int(y.value_counts().min()))
    if folds < 2:
        return None
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)

    results = []
    for name, clf in [
        ("Logistic_L2",  LogisticRegression(C=1.0, max_iter=5000,
                                             class_weight="balanced",
                                             solver="liblinear", random_state=42)),
        ("Ridge_logit",  LogisticRegression(C=0.1, max_iter=5000,
                                             class_weight="balanced",
                                             solver="liblinear", random_state=42)),
        ("LASSO_logit",  LogisticRegression(C=0.1, penalty="l1", max_iter=5000,
                                             class_weight="balanced",
                                             solver="liblinear", random_state=42)),
    ]:
        try:
            y_prob = cross_val_predict(clf, X_sc, y, cv=cv, method="predict_proba")[:,1]
            auc    = roc_auc_score(y, y_prob)
            clf.fit(X_sc, y)
            coefs = clf.coef_[0]
            results.append({
                "model": name, "auc": round(auc,4), "n": len(y),
                **{feat_cols[i]: round(float(coefs[i]),5) for i in range(len(feat_cols))}
            })
        except Exception as e:
            results.append({"model": name, "error": str(e)})

    return pd.DataFrame(results)


print("Section C: Building outcome feature matrices and running regularised models...")

section_c_results = {}

for ds, df in all_data.items():
    if "outcome_binary" not in df.columns:
        print(f"  {ds}: no outcome — skipping")
        continue

    has_outcome = df.groupby(ID_COL)["outcome_binary"].first().notna().sum()
    if has_outcome < 20:
        print(f"  {ds}: too few outcomes ({has_outcome}) — skipping")
        continue

    ds_dir  = OUTPUT_ROOT / ds
    ds_dir.mkdir(exist_ok=True)
    emo = all_emotions[ds]

    feat_df = build_outcome_features(df, emo)
    feat_df.to_csv(ds_dir / "C_outcome_features.csv", index=False)

    reg_df  = run_regularised_models(feat_df)

    if reg_df is not None:
        reg_df.to_csv(ds_dir / "C_regularised_model_results.csv", index=False)
        section_c_results[ds] = {"feat_df": feat_df, "reg_df": reg_df}
        for _, row in reg_df.iterrows():
            print(f"  {ds} | {row['model']}: AUC={row.get('auc','—')}")
    else:
        print(f"  {ds}: model fitting failed (insufficient variance or N)")


Section C: Building outcome feature matrices and running regularised models...
  dealerships | Logistic_L2: AUC=0.9617
  dealerships | Ridge_logit: AUC=0.9617
  dealerships | LASSO_logit: AUC=0.5548
  heddaya: too few outcomes (0) — skipping
  casino: model fitting failed (insufficient variance or N)
  a2a | Logistic_L2: AUC=0.9635
  a2a | Ridge_logit: AUC=0.9673
  a2a | LASSO_logit: AUC=0.9618
  emaad | Logistic_L2: AUC=0.9918
  emaad | Ridge_logit: AUC=0.9932
  emaad | LASSO_logit: AUC=0.9754


In [56]:
# ─── Top LASSO coefficients plot ─────────────────────────────────────────────
# Shows which emotion × role × timing features drive outcome prediction

fig, axes = plt.subplots(1, len(section_c_results), figsize=(8*len(section_c_results), 8),
                          constrained_layout=True)
if len(section_c_results) == 1:
    axes = [axes]

for ax, (ds, res) in zip(axes, section_c_results.items()):
    reg_df = res["reg_df"]
    lasso  = reg_df[reg_df["model"]=="LASSO_logit"]
    if lasso.empty:
        continue

    # Select non-zero coefficients (LASSO sets unimportant ones to 0)
    row    = lasso.iloc[0]
    coef_cols = [c for c in reg_df.columns
                 if c not in {"model","auc","n","error"}
                 and pd.notna(row.get(c)) and abs(float(row.get(c,0))) > 1e-6]

    if not coef_cols:
        ax.text(0.5, 0.5, "No non-zero LASSO coefficients", ha="center", va="center")
        continue

    coef_vals = [(c, float(row[c])) for c in coef_cols]
    coef_vals.sort(key=lambda x: abs(x[1]), reverse=True)
    top_coefs = coef_vals[:20]

    labels = [c for c,_ in top_coefs]
    values = [v for _,v in top_coefs]
    colors = [DS_COLORS.get(ds,"#888") if v > 0 else "#E05A47" for v in values]

    ax.barh(labels, values, color=colors, alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("LASSO coefficient (standardised features)", fontsize=9)
    auc_val = row.get("auc","—")
    ax.set_title(f"{ds}\nLASSO logistic regression\nCV AUC = {auc_val}\n"
                 f"(colored=positive predictor, red=negative)",
                 fontsize=9, fontweight="bold")
    ax.tick_params(labelsize=7)

fig.suptitle("Section C — LASSO Feature Selection: Top Emotional Predictors of Outcome\n"
             "Zero coefficients excluded — these are the emotions that survive feature selection",
             fontsize=10, fontweight="bold")
fig.savefig(OUTPUT_ROOT / "C1_lasso_top_features.png", dpi=150, bbox_inches="tight")
plt.close()
print("✓ Saved: C1_lasso_top_features.png")


✓ Saved: C1_lasso_top_features.png


In [57]:
# ─── Ridge vs LASSO vs Logistic AUC comparison ───────────────────────────────

if section_c_results:
    rows = []
    for ds, res in section_c_results.items():
        for _, row in res["reg_df"].iterrows():
            rows.append({"dataset": ds, "model": row.get("model",""),
                         "auc": row.get("auc", np.nan)})
    auc_df = pd.DataFrame(rows).dropna(subset=["auc"])

    fig, ax = plt.subplots(figsize=(8, 4))
    ds_list  = auc_df["dataset"].unique()
    mod_list = auc_df["model"].unique()
    x = np.arange(len(ds_list))
    w = 0.7 / max(len(mod_list), 1)

    for i, model in enumerate(mod_list):
        sub    = auc_df[auc_df["model"]==model]
        aucs   = [sub[sub["dataset"]==ds]["auc"].values[0]
                  if not sub[sub["dataset"]==ds].empty else np.nan
                  for ds in ds_list]
        offset = (i - len(mod_list)/2 + 0.5) * w
        ax.bar(x + offset, aucs, w, label=model, alpha=0.85)

    ax.axhline(0.5, color="red", linestyle="--", linewidth=0.9, label="Chance")
    ax.set_xticks(x)
    ax.set_xticklabels(ds_list, fontsize=10)
    ax.set_ylabel("CV AUC (5-fold stratified)", fontsize=10)
    ax.set_title("Section C — Regularised Model AUC Comparison\n"
                 "(Logistic L2, Ridge logit, LASSO logit)", fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    plt.tight_layout()
    fig.savefig(OUTPUT_ROOT / "C2_model_auc_comparison.png", dpi=150)
    plt.close()
    print("✓ Saved: C2_model_auc_comparison.png")


✓ Saved: C2_model_auc_comparison.png


---
## Section D — Emotional symmetry: buyer-only, seller-only, and difference models

**Core question:** Does combining buyer and seller emotions help prediction, or does one role dominate?

We test four model variants (per the advisor's recommendation):
- **Model A** — Buyer emotions only
- **Model B** — Seller emotions only
- **Model C** — Emotion difference (buyer − seller) only
- **Model D** — Buyer + Seller combined (with Ridge to handle collinearity)

*Logic: If buyer-only AUC ≈ combined AUC, the seller's emotional contribution
is redundant — buyers are driving the emotional channel. If the difference model
performs well, asymmetry itself is informative.*


In [58]:
def model_variant_auc(feat_df: pd.DataFrame, feature_filter: str,
                       outcome_col: str = "outcome_binary",
                       penalty: str = "l2", C: float = 1.0) -> Optional[float]:
    """
    Fit one model variant and return CV AUC.
    feature_filter: 'buyer' | 'seller' | 'diff' | 'all'
    """
    valid  = feat_df.dropna(subset=[outcome_col]).copy()
    y_raw  = pd.to_numeric(valid[outcome_col], errors="coerce")
    keep   = y_raw.isin([0,1])
    valid  = valid.loc[keep].copy()
    y      = y_raw.loc[keep].astype(int)

    if len(valid) < 20 or y.nunique() < 2 or y.value_counts().min() < 3:
        return None

    if feature_filter == "buyer":
        fcols = [c for c in valid.columns if "_b_" in c]
    elif feature_filter == "seller":
        fcols = [c for c in valid.columns if "_s_" in c]
    elif feature_filter == "diff":
        fcols = [c for c in valid.columns if "_diff_" in c]
    else:  # all
        fcols = [c for c in valid.columns
                 if c not in {ID_COL, outcome_col, "n_turns"}
                 and pd.api.types.is_numeric_dtype(valid[c])]

    if not fcols:
        return None

    X = valid[fcols].replace([np.inf,-np.inf], np.nan).fillna(0)
    if X.shape[1] == 0:
        return None

    folds = min(N_FOLDS, int(y.value_counts().min()))
    if folds < 2:
        return None

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("logit",  LogisticRegression(C=C, penalty=penalty, max_iter=5000,
                                       class_weight="balanced",
                                       solver="liblinear", random_state=42)),
    ])
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    try:
        y_prob = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:,1]
        return round(roc_auc_score(y, y_prob), 4)
    except Exception:
        return None


print("Section D: Running buyer-only / seller-only / difference / combined models...")

section_d_results = {}

for ds, res in section_c_results.items():
    feat_df = res["feat_df"]
    row     = {"dataset": ds}

    for variant, filt, pen, C in [
        ("A_buyer_only",    "buyer",  "l2", 1.0),
        ("B_seller_only",   "seller", "l2", 1.0),
        ("C_diff_b_minus_s","diff",   "l2", 1.0),
        ("D_combined_ridge","all",    "l2", 0.1),  # Ridge via low C
        ("D_combined_lasso","all",    "l1", 0.1),  # LASSO
    ]:
        auc = model_variant_auc(feat_df, filt, penalty=pen, C=C)
        row[variant] = auc
        print(f"  {ds} | {variant}: AUC={auc}")

    section_d_results[ds] = row

# Save summary
d_df = pd.DataFrame(section_d_results.values()).set_index("dataset")
d_df.to_csv(OUTPUT_ROOT / "D_symmetry_model_comparison.csv")
print("\n✓ Section D complete.")


Section D: Running buyer-only / seller-only / difference / combined models...
  dealerships | A_buyer_only: AUC=0.8087
  dealerships | B_seller_only: AUC=0.7809
  dealerships | C_diff_b_minus_s: AUC=0.5826
  dealerships | D_combined_ridge: AUC=0.8417
  dealerships | D_combined_lasso: AUC=0.4939
  a2a | A_buyer_only: AUC=0.9588
  a2a | B_seller_only: AUC=0.8495
  a2a | C_diff_b_minus_s: AUC=0.7783
  a2a | D_combined_ridge: AUC=0.9603
  a2a | D_combined_lasso: AUC=0.9602
  emaad | A_buyer_only: AUC=0.9912
  emaad | B_seller_only: AUC=0.9471
  emaad | C_diff_b_minus_s: AUC=0.9616
  emaad | D_combined_ridge: AUC=0.9808
  emaad | D_combined_lasso: AUC=0.9656

✓ Section D complete.


In [59]:
# ─── Model variant comparison plot ───────────────────────────────────────────
if section_d_results:
    d_df_plot = pd.DataFrame(section_d_results.values()).set_index("dataset")
    d_df_plot.columns = [c.replace("_"," ").replace("A ","A: ").replace("B ","B: ")
                          .replace("C ","C: ").replace("D ","D: ")
                          for c in d_df_plot.columns]

    fig, ax = plt.subplots(figsize=(11, 5))
    ds_list  = d_df_plot.index.tolist()
    mod_cols = d_df_plot.columns.tolist()
    x = np.arange(len(ds_list))
    w = 0.8 / max(len(mod_cols), 1)
    palette = ["#4C9EEB","#F0956A","#1A8A7A","#E05A47","#7B68C8"]

    for i, col in enumerate(mod_cols):
        vals   = d_df_plot[col].values
        offset = (i - len(mod_cols)/2 + 0.5) * w
        ax.bar(x + offset, [v if pd.notna(v) else 0 for v in vals],
               w, label=col, alpha=0.85, color=palette[i % len(palette)])
        for j, v in enumerate(vals):
            if pd.notna(v):
                ax.text(x[j] + offset, v + 0.005, f"{v:.3f}",
                        ha="center", va="bottom", fontsize=6, rotation=45)

    ax.axhline(0.5, color="red", linestyle="--", linewidth=0.9, label="Chance (0.5)")
    ax.set_xticks(x)
    ax.set_xticklabels(ds_list, fontsize=10)
    ax.set_ylabel("CV AUC (5-fold)", fontsize=10)
    ax.set_title("Section D — Model Variants: Buyer-only vs Seller-only vs Difference vs Combined\n"
                 "Which role's emotional dynamics drive outcome prediction?", fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7.5, bbox_to_anchor=(1.01,1), loc="upper left")
    plt.tight_layout()
    fig.savefig(OUTPUT_ROOT / "D1_model_variant_comparison.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Saved: D1_model_variant_comparison.png")


✓ Saved: D1_model_variant_comparison.png


---
## Final — Master summary across all sections

In [60]:
print("=" * 70)
print("STAGE 1.5 DYNAMIC VALIDATION — MASTER SUMMARY")
print("=" * 70)

for ds in all_data:
    print(f"\n{'─'*50}")
    print(f"DATASET: {ds}")
    print(f"{'─'*50}")

    # Section A
    if ds in section_a_results:
        ra = section_a_results[ds]
        n_stat   = ra["stationary"].sum()
        n_struct = ra["structured"].sum()
        mean_r2  = ra["ar1_r2_in"].mean()
        print(f"  [A] Temporal structure: {n_stat}/{len(ra)} stationary | "
              f"{n_struct}/{len(ra)} AR beats RW | mean AR1 R²={mean_r2:.3f}")
        top_emo = ra.nlargest(3,"ar1_r2_in")["emotion"].tolist()
        print(f"      Most predictable emotions: {top_emo}")

    # Section B
    if ds in section_b_results:
        rb = section_b_results[ds]
        nb = (rb["leader"]=="buyer").sum()
        ns = (rb["leader"]=="seller").sum()
        ny = (rb["leader"]=="symmetric").sum()
        ngc_b2s = rb["gc_buyer2seller_sig"].sum()
        ngc_s2b = rb["gc_seller2buyer_sig"].sum()
        print(f"  [B] Directional influence: buyer leads={nb} | seller leads={ns} | symmetric={ny}")
        print(f"      Granger significant: buyer→seller={ngc_b2s} | seller→buyer={ngc_s2b} emotions")

    # Section C
    if ds in section_c_results:
        rc = section_c_results[ds]["reg_df"]
        for _, row in rc.iterrows():
            print(f"  [C] {row.get('model','?')}: AUC={row.get('auc','—')}")

    # Section D
    if ds in section_d_results:
        rd = section_d_results[ds]
        print(f"  [D] Buyer-only AUC:  {rd.get('A_buyer_only','—')}")
        print(f"      Seller-only AUC: {rd.get('B_seller_only','—')}")
        print(f"      Diff AUC:        {rd.get('C_diff_b_minus_s','—')}")
        print(f"      Combined-Ridge:  {rd.get('D_combined_ridge','—')}")

print(f"\n✓ All outputs saved to: {OUTPUT_ROOT}")
print("\nOutput files:")
for f in sorted(OUTPUT_ROOT.glob("*.png")) + sorted(OUTPUT_ROOT.glob("*.csv")):
    print(f"  {f.name}")


STAGE 1.5 DYNAMIC VALIDATION — MASTER SUMMARY

──────────────────────────────────────────────────
DATASET: dealerships
──────────────────────────────────────────────────
  [A] Temporal structure: 27/27 stationary | 27/27 AR beats RW | mean AR1 R²=0.003
      Most predictable emotions: ['gratitude', 'optimism', 'pride']
  [B] Directional influence: buyer leads=3 | seller leads=5 | symmetric=19
      Granger significant: buyer→seller=2 | seller→buyer=4 emotions
  [C] Logistic_L2: AUC=0.9617
  [C] Ridge_logit: AUC=0.9617
  [C] LASSO_logit: AUC=0.5548
  [D] Buyer-only AUC:  0.8087
      Seller-only AUC: 0.7809
      Diff AUC:        0.5826
      Combined-Ridge:  0.8417

──────────────────────────────────────────────────
DATASET: heddaya
──────────────────────────────────────────────────
  [A] Temporal structure: 27/27 stationary | 27/27 AR beats RW | mean AR1 R²=0.004
      Most predictable emotions: ['joy', 'excitement', 'admiration']
  [B] Directional influence: buyer leads=1 | seller le